# 📡 NBL Model Evaluation
Load a pre-trained NBL model and evaluate on a freshly generated eval dataset.

**Run all cells top to bottom. After Cell 1 (install), restart the kernel, then continue from Cell 2.**

## 1 · Install Dependencies
Run once, then restart kernel.

In [1]:
# ============================================================
# Step 1 of 3: Install packages — Run this cell FIRST
# After this cell finishes, proceed to the patch cell below.
# Do NOT import sionna anywhere until after the patch cell runs.
# ============================================================
import subprocess, sys

def pip(*args, no_deps=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet']
    if no_deps:
        cmd.append('--no-deps')
    cmd.extend(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f'OK      {args[0]}' if result.returncode == 0
          else f'FAILED  {args[0]}:\n{result.stderr[-400:]}')

pip('numpy')
pip('scipy')
pip('pandas')
pip('matplotlib')
pip('seaborn')
pip('keras')
pip('scikit-learn')
pip('statsmodels')
pip('h5py')
pip('plotly')
pip('numba')
pip('pythreejs')
pip('tensorflow>=2.16,<2.20')
pip('mitsuba==3.4.1')
pip('drjit==0.4.4')
pip('tensorflow-probability')
# --no-deps bypasses sionna's stale 'tensorflow<2.14' metadata constraint.
# The actual runtime code is fully compatible with TF 2.16+.
# Do NOT upgrade to sionna>=1.0 — it has an entirely different API.
pip('sionna==0.19.2', no_deps=True)
print('Installation complete.')


OK      numpy
OK      scipy
OK      pandas
OK      matplotlib
OK      seaborn
OK      keras
OK      scikit-learn
OK      statsmodels
OK      h5py
OK      plotly
OK      numba
OK      pythreejs
OK      tensorflow>=2.16,<2.20
OK      mitsuba==3.4.1
OK      drjit==0.4.4
OK      tensorflow-probability
OK      sionna==0.19.2
Installation complete.


## 2 · Compatibility Patches
Must run **before** any `import sionna`.

In [2]:
!pip install importlib_resources
import importlib.util
import pathlib
import sys
import numpy as np

# ── Patch 1: numpy.infty removed in NumPy 2.x ──────────────────────────────
if not hasattr(np, "infty"):
    np.infty = np.inf

# ── Patch 2: sionna/rt/itu_materials.py circular-import (Python 3.12+) ──────
spec = importlib.util.find_spec("sionna")
if spec is None:
    raise RuntimeError("sionna not found — please run the install cell above.")

sionna_root = pathlib.Path(spec.origin).parent
itu_path    = sionna_root / "rt" / "itu_materials.py"
src         = itu_path.read_text()

MARKER = "# PATCHED_FOR_PY312_CIRCULAR_IMPORT"
if MARKER not in src:
    lazy_helper = (
        MARKER + "\n"
        "def _get_scene_class():\n"
        "    from sionna.rt.scene import Scene  # noqa: PLC0415\n"
        "    return Scene\n\n"
    )
    itu_path.write_text(lazy_helper + src.replace("scene.Scene()", "_get_scene_class()()"))
    print(f"Patched: {itu_path}")

# Clear any stale partial sionna imports
for k in [k for k in sys.modules if k.startswith("sionna")]:
    del sys.modules[k]

# ── Patch 3: tf.grad_pass_through removed in TF 2.16+ ───────────────────────
import tensorflow as tf
if not hasattr(tf, "grad_pass_through"):
    def _grad_pass_through(fn):
        @tf.custom_gradient
        def wrapper(*args, **kwargs):
            result = fn(*args, **kwargs)
            def grad(*dy):
                return dy if len(dy) > 1 else dy[0]
            return result, grad
        return wrapper
    tf.grad_pass_through = _grad_pass_through
    print("[Compat] tf.grad_pass_through patched for TF >= 2.16.")

import sionna
print(f"TensorFlow : {tf.__version__}")
print(f"Sionna     : {sionna.__version__}")
print("Patches applied. Ready.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sionna 0.19.2 requires ipydatawidgets==4.3.2, but you have ipydatawidgets 4.3.5 which is incompatible.
sionna 0.19.2 requires jupyterlab-widgets==3.0.5, but you have jupyterlab-widgets 3.0.16 which is incompatible.
sionna 0.19.2 requires tensorflow<2.16.0,>=2.13.0, but you have tensorflow 2.19.1 which is incompatible.
Patched: /home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sionna/rt/itu_materials.py


2026-05-21 21:57:32.452471: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-21 21:57:32.457819: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-21 21:57:32.493435: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-21 21:57:32.528368: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779400652.561474   14454 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779400652.57

TensorFlow : 2.19.1
Sionna     : 0.19.2
Patches applied. Ready.


## 3 · TF Compatibility Shims

In [3]:
# ============================================================
# Step 3 of 3: TF 2.16+ compatibility shims
# ============================================================
import tensorflow as tf

# tf.grad_pass_through was removed in TF 2.16.
# Re-implement as a straight-through gradient estimator via
# tf.custom_gradient (the recommended TF 2.16+ approach).
if not hasattr(tf, 'grad_pass_through'):
    def _grad_pass_through(fn):
        """Straight-through estimator — drop-in for tf.grad_pass_through.
        Forward: runs fn normally.
        Backward: identity (gradients pass through unchanged).
        """
        @tf.custom_gradient
        def wrapper(*args, **kwargs):
            result = fn(*args, **kwargs)
            def grad(*dy):
                return dy if len(dy) > 1 else dy[0]
            return result, grad
        return wrapper
    tf.grad_pass_through = _grad_pass_through
    print('[Compat] tf.grad_pass_through patched for TF >= 2.16.')
else:
    print('[Compat] tf.grad_pass_through already present.')

# Now safe to import sionna
import sionna
print(f'TensorFlow : {tf.__version__}')
print(f'Sionna     : {sionna.__version__}')
print('Ready.')


[Compat] tf.grad_pass_through already present.
TensorFlow : 2.19.1
Sionna     : 0.19.2
Ready.


In [4]:
import os
# ============================================================
# Cell 4 · Configuration  — EDIT THIS CELL
# ============================================================

# ── Output / working directory ──────────────────────────────────────────────
#   Google Colab  : '/content/'  or  '/content/drive/MyDrive/nbl_eval/'
#   Local / other : any writable path, e.g. './nbl_eval/'
OUTPUT_DIR = '/teamspace/studios/this_studio/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Pre-generated eval dataset ──────────────────────────────────────────────
#   Set to the full path of your .pickle eval file.
#   Example:  '/content/drive/MyDrive/eval_6pg_scene_A_2000_inputs_outputs_16_16_16_16_16.pickle'
#   If you don't have one, set GENERATE_DATASET = True below.
EVAL_PICKLE_PATH = os.path.join(OUTPUT_DIR, 'eval_6pg_scene_A_2000_inputs_outputs_16_16_16_16_16.pickle')

# ── Dataset generation (only needed if you have no .pickle yet) ─────────────
GENERATE_DATASET = False   # set True to run ray-tracing + dataset build
EVAL_SIZE        = 2_000   # number of eval samples to generate
UE_POOL_SIZE     = 1_000   # number of UEs to ray-trace
RT_BATCH_SIZE    = 80      # ray-tracing batch size

# ── Evaluation batch size ────────────────────────────────────────────────────
BATCH_SIZE = 64            # samples per evaluation batch (lower if OOM)
NCSI       = 16            # CSI-RS subset size per cell

SCENE = 'A'                # 'A' = Munich 10 GHz

print('Config:')
print(f'  OUTPUT_DIR        : {OUTPUT_DIR}')
print(f'  EVAL_PICKLE_PATH  : {EVAL_PICKLE_PATH}')
print(f'  GENERATE_DATASET  : {GENERATE_DATASET}')
print(f'  BATCH_SIZE        : {BATCH_SIZE}')


Config:
  OUTPUT_DIR        : /teamspace/studios/this_studio/outputs
  EVAL_PICKLE_PATH  : /teamspace/studios/this_studio/outputs/eval_6pg_scene_A_2000_inputs_outputs_16_16_16_16_16.pickle
  GENERATE_DATASET  : False
  BATCH_SIZE        : 64


## 5 · System Constants

In [8]:
pi    = np.pi
ctype = np.complex64

# ── Antenna / array geometry ────────────────────────────────────────────────
FC    = 10e9          # carrier frequency (Hz) — Scene A
Nx    = 16            # horizontal antenna elements
Ny    = 16            # vertical antenna elements
Nt    = Nx * Ny       # total TX antennas (256)
Nr    = 4             # RX antennas per UE
C     = 3             # number of cells / base-stations

# ── Beamspace / codebook dimensions ─────────────────────────────────────────
Nx1   = 16
Ny1   = 16
Lmax  = 16            # max SSB beams per cell
N_CSI = 8             # CSI-RS beams
Bg    = 4             # beam groups

# ── OFDM / channel parameters ────────────────────────────────────────────────
SUBCARRIER_SPACING = 30e3 * 12
K_TOTAL            = 20
T_SLOTS            = 1
PATH_GAIN_EXP      = 6
NOISE_FIGURE_DB    = 7.0
NOISE_BW           = K_TOTAL * SUBCARRIER_SPACING

# ── Dataset sizes ────────────────────────────────────────────────────────────
DATASET_SIZE_TRAIN = 20_000
DATASET_SIZE_EVAL  = 2_000
UMAX               = 20
UMIN               = 8
NEW_USER_PROB      = 0.2
B_PHASE            = 3

# ── Angular scan range ───────────────────────────────────────────────────────
THETA_MIN = 3 * pi / 24
THETA_MAX = 21 * pi / 24
PHI_MIN   = 3 * pi / 24
PHI_MAX   = 21 * pi / 24

## 6 · Beamspace Transformation Matrices

In [9]:
def build_steering_vector(N, theta):
    n = np.arange(N, dtype=np.float32)
    return (1.0 / np.sqrt(N)) * np.exp(1j * pi * n * theta).astype(ctype)


def build_transformation_matrix(N, n_samples, theta_min=0.0, theta_max=1.0):
    assert n_samples >= N
    thetas = np.linspace(theta_min, theta_max, num=n_samples, dtype=np.float32)
    U = np.stack([build_steering_vector(N, th) for th in thetas], axis=1)
    return U.astype(ctype)


U_NxNx     = build_transformation_matrix(Nx, Nx1, THETA_MIN, THETA_MAX)
U_NyNy     = build_transformation_matrix(Ny, Ny1, PHI_MIN,   PHI_MAX)
U_NxNx_inv = np.linalg.pinv(U_NxNx).astype(ctype)
U_NyNy_inv = np.linalg.pinv(U_NyNy).astype(ctype)

print(f"U_NxNx : {U_NxNx.shape}")
print(f"U_NyNy : {U_NyNy.shape}")

U_NxNx : (16, 16)
U_NyNy : (16, 16)


## 7 · DFT Codebook

In [10]:
# ──────────────────────────────────────────────────────────────────────────────
# DFT CODEBOOK BASELINE
# ──────────────────────────────────────────────────────────────────────────────

def gen_codebook(Nx, Ny, OH=1, OV=1):                                                                
    """                                                                                              
    Generate a full oversampled 2-D DFT codebook.

    Args:
        Nx, Ny : physical array dimensions
        OH, OV : horizontal / vertical oversampling factors

    Returns:
        codebook : shape [Nx*OH*Ny*OV, Nt]  complex64
    """
    Nt = Nx * Ny
    DFT_y = np.fft.fft(np.eye(Ny * OV))[:, :Ny] / np.sqrt(Ny)    # [Ny*OV, Ny]
    DFT_x = np.fft.fft(np.eye(Nx * OH))[:Nx, :]  / np.sqrt(Nx)   # [Nx, Nx*OH]

    all_cb = np.zeros((Nx * OH * Ny * OV, Nx, Ny), dtype=ctype)
    for i in range(OH * Nx):
        for j in range(OV * Ny):
            all_cb[i * OV * Ny + j, :, :] = DFT_x[:, i:i+1] @ DFT_y[j:j+1, :]
    return all_cb.reshape(Nx * OH * Ny * OV, Nt).astype(ctype)


def DFT_codebook(n_beams=8, Nx=Nx, Ny=Ny, OH=2, OV=2, **kwargs):                 # Only one usage below in (## 17 · Neural Network Architecture & NBL Model  *(paper Section IV, Figure 3)*)
                                                                                 # Build DFT codebooks for comparison
                                                                                 # dft_ssb_cb = DFT_codebook(n_beams=Lmax, Nx=Nx, Ny=Ny, OH=2, OV=2)
                                                                                 #  dft_csi_cb = DFT_codebook(n_beams=Lmax * N_csi, Nx=Nx, Ny=Ny, OH=N_csi, OV=N_csi)
    """
    Generate an n_beams-entry DFT codebook for an Nx×Ny planar array.

    When n_beams >= Nx*Ny*OH*OV the full oversampled DFT grid is used and
    then down-selected.  For smaller codebooks a centred sub-array DFT is
    used so the beams cover the most relevant angular region.

    Args:
        n_beams   : number of beamformers to return
        Nx, Ny    : physical array dimensions
        OH, OV    : oversampling (default 2×2, matching paper Section V)

    Returns:
        codebook  : shape [n_beams, Nt, 1]  complex64
                    (last dim kept for legacy callers that expect the [Nt,1] shape)
    """
    Nt = Nx * Ny
    print(f"Generating DFT codebook: {n_beams} beams for {Nx}×{Ny} array "
          f"(OH={OH}, OV={OV})")

    if n_beams >= Nx * Ny * OH * OV:
        # Full oversampled DFT — take the first n_beams rows
        cb = gen_codebook(Nx=Nx, Ny=Ny, OH=OH, OV=OV)[:n_beams]   # [n_beams, Nt]
        cb = cb.reshape(n_beams, Nt, 1)

        
    else:                                                                                   ##### check نفس الحاجة؟
        # Centred sub-array DFT for compact codebooks
        cb = np.zeros((n_beams, Nx, Ny), dtype=ctype)
        if n_beams <= 8:
            Nx1_ = 2
        elif n_beams <= 32:
            Nx1_ = 4
        else:
            Nx1_ = 8
        Ny1_ = n_beams // Nx1_

        DFT_x = np.fft.fft(np.eye(Nx1_)) / np.sqrt(Nx1_)
        DFT_y = np.fft.fft(np.eye(Ny1_)) / np.sqrt(Ny1_)

        x_lo = Nx // 2 - Nx1_ // 2
        y_lo = Ny // 2 - Ny1_ // 2
        for nx in range(Nx1_):
            for ny in range(Ny1_):
                cb[nx * Ny1_ + ny,
                   x_lo:x_lo + Nx1_,
                   y_lo:y_lo + Ny1_] = DFT_x[:, nx:nx+1] @ DFT_y[ny:ny+1, :]
        cb = cb.reshape(n_beams, Nt, 1)

    # Normalise each beam to unit-power × sqrt(Nt)  (matches paper notation)
    cb = cb / (np.linalg.norm(cb, axis=1, keepdims=True) + 1e-10) * np.sqrt(Nt)
    return cb

## 8 · Scene Setup

In [11]:
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray


def setup_scene_munich():
    scene = load_scene(sionna.rt.scene.munich)
    scene.frequency = FC


    for name in list(scene.transmitters.keys()):
        scene.remove(name)
    for name in list(scene.receivers.keys()):
        scene.remove(name)
    scene.add(Transmitter(name="tx1", position=[8.5,  21,  40], orientation=[1.25, 0, 0]))
    scene.add(Transmitter(name="tx2", position=[280,275,40], orientation=[-2*np.pi/3-0.25, 0, 0]))
    scene.add(Transmitter(name="tx3", position=[-292,295,40], orientation=[-np.pi/3+0.25, 0, 0]))
    scene.synthetic_array = True


    tx1_pos = np.array([8.5,   21,  40])
    tx2_pos = np.array([280,   275, 40])
    tx3_pos = np.array([-292,  295, 40])

    
    isd = np.mean([
    np.linalg.norm(tx1_pos - tx2_pos),
    np.linalg.norm(tx1_pos - tx3_pos),
    np.linalg.norm(tx2_pos - tx3_pos)
    ])
    print(f"ISD = {isd:.1f}m")



    scene.tx_array = PlanarArray(num_rows=1,
                                num_cols=1,
                                vertical_spacing=0.5,
                                horizontal_spacing=0.5,
                                pattern="tr38901",
                                polarization="V")

    # Configure antenna array for all receivers
    scene.rx_array = PlanarArray(num_rows=1,
                                num_cols=1,
                                vertical_spacing=0.5,
                                horizontal_spacing=0.5,
                                pattern="iso",
                                polarization="V")
        


    

    cm_size = (600,400.)
    center_loc = np.array(np.mean([tx1_pos, tx2_pos, tx3_pos], axis=0))
    center_loc[-1] = 1.5

    cm = scene.coverage_map(
        max_depth=5,
        cm_cell_size=(5., 5.),
        cm_size=cm_size,
        cm_center=center_loc,
        cm_orientation=(0., 0., 0.),
        num_samples=int(10e6),
        diffraction=True,
        scattering=False
    )
    return scene, cm

## 9 · UE Channel Generation

In [12]:

def _remove_all_ues(scene):
    for name in list(scene.receivers.keys()):
        scene.remove(name)
    return scene


def _add_ues(scene, n):
    from sionna.rt import Receiver
    for i in range(n):
        scene.add(Receiver(name=f"rx{i:04d}", position=[0.0, 0.0, 0.0]))
    return scene


def scene2channels(scene, T=1, K=K_TOTAL):
    from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies
    frequencies = subcarrier_frequencies(K, SUBCARRIER_SPACING)
    vx, vy = np.random.randn(2) * 2
    paths  = scene.compute_paths(max_depth=5, num_samples=int(1e6), diffraction=True, scattering=False)
    n_rx   = len(scene.receivers)
    rx_vel = np.zeros((n_rx, 3), dtype=np.float32)
    rx_vel[:, 0] = vx
    rx_vel[:, 1] = vy
    paths.apply_doppler(sampling_frequency=1e3, num_time_steps=T, rx_velocities=rx_vel)
    a, tau   = paths.cir()
    h_freq   = cir_to_ofdm_channel(frequencies, a, tau, normalize=False)
    
    h_freq   = h_freq[0] 
    h_freq   = tf.transpose(h_freq, [0, 2, 1, 3, 4, 5])
    return h_freq

x_pos_offsets = np.array([[0, 0, 0.5]], dtype=np.float32)

def _sample_positions_to_float(sampled_pos):
    """Handle Sionna 0.19.2 sample_positions() output: [num_tx, num_pos, 3] + cell_ids tuple."""
    # Unpack tuple — take positions only, discard cell indices
    if isinstance(sampled_pos, (tuple, list)):
        sampled_pos = sampled_pos[0]   # [num_tx, num_pos, 3]
    sampled_pos = tf.cast(sampled_pos, tf.float32)
    # Reshape [num_tx, num_pos, 3] → [num_tx * num_pos, 3]  — keep ALL TX samples
    sampled_pos = tf.reshape(sampled_pos, [-1, 3])
    return sampled_pos   # [num_tx * num_pos, 3]


def generate_ue_channels(scene, cm, dataset_size, batch_size=80, T=T_SLOTS, K=K_TOTAL):
    print(f"  Array config: Nr={Nr}, Nt={Nt}, C={C}")
    channel_pool   = np.zeros((dataset_size, C, Nr, Nt, T, K), dtype=ctype)
    ue_positions   = np.zeros((dataset_size, 3), dtype=np.float32)
    scene          = _remove_all_ues(scene)
    scene          = _add_ues(scene, batch_size)
    rx_names       = list(scene.receivers.keys())
    n_loops        = int(np.ceil(dataset_size / batch_size))
    batch_sizes    = np.full(n_loops, batch_size, dtype=int)
    batch_sizes[-1] = dataset_size - batch_size * (n_loops - 1)
    missed_indices = []
    ds_counter     = 0

    for loop_idx in range(n_loops):
        active_b = batch_sizes[loop_idx]
        print(f"  Generating channels {ds_counter}–{ds_counter + active_b} / {dataset_size}")

        # ── sample positions — request ceil(active_b/C) per TX → reshape to [C*n, 3] → take active_b
        n_per_tx    = int(np.ceil(active_b / C))
        sampled_pos = cm.sample_positions(n_per_tx, min_val_db=-100.0, min_dist=20.0)
        sampled_pos = _sample_positions_to_float(sampled_pos)  # [C * n_per_tx, 3]
        sampled_pos = sampled_pos[:active_b]                   # trim to exact active_b

        # add small height offset like author
        offset      = (tf.random.uniform(shape=(active_b, 3), dtype=tf.float32) - 0.5) * x_pos_offsets
        sampled_pos = sampled_pos + offset

        ue_positions[ds_counter:ds_counter + active_b] = sampled_pos.numpy()

        for i in range(active_b):
            scene.receivers[rx_names[i]].position = sampled_pos[i]
        if active_b < len(rx_names):
            for name in rx_names[active_b:]:
                scene.remove(name)

        h = scene2channels(scene, T=T, K=K).numpy()
        h = h * (10 ** PATH_GAIN_EXP)
        channel_pool[ds_counter:ds_counter + active_b] = h
        h_check   = h[:, :, :, :, 0, 0]
        svd_check = np.linalg.svd(h_check.reshape(active_b, C, Nr, Nt), compute_uv=False)
        zero_power = np.where(np.sum(svd_check, axis=(1, 2)) < 1e-10)[0] + ds_counter
        missed_indices.extend(zero_power.tolist())
        ds_counter += active_b

        if active_b < batch_size:
            scene    = _remove_all_ues(scene)
            scene    = _add_ues(scene, batch_size)
            rx_names = list(scene.receivers.keys())

    if missed_indices:
        print(f"  Retrying {len(missed_indices)} failed positions...")
        n_missed  = len(missed_indices)
        scene     = _remove_all_ues(scene)
        scene     = _add_ues(scene, n_missed)
        rx_names  = list(scene.receivers.keys())

        n_per_tx    = int(np.ceil(n_missed / C))
        sampled_pos = cm.sample_positions(n_per_tx, min_val_db=-120.0, min_dist=20.0)
        sampled_pos = _sample_positions_to_float(sampled_pos)
        sampled_pos = sampled_pos[:n_missed]
        offset      = (tf.random.uniform(shape=(n_missed, 3), dtype=tf.float32) - 0.5) * x_pos_offsets
        sampled_pos = sampled_pos + offset

        for i in range(n_missed):
            scene.receivers[rx_names[i]].position = sampled_pos[i]
            ue_positions[missed_indices[i]] = sampled_pos[i].numpy()

        h_retry = scene2channels(scene, T=T, K=K).numpy() * (10 ** PATH_GAIN_EXP)
        for i, idx in enumerate(missed_indices):
            channel_pool[idx] = h_retry[i]

    print(f"  Done. channel_pool shape: {channel_pool.shape}")
    return channel_pool, ue_positions



'''
# After one batch of sample_positions, print the XY distribution:
sampled_pos = cm.sample_positions(200, min_val_db=-100.0, min_dist=20.0)
pos = np.array(sampled_pos[0][0], dtype=np.float32)  # or however your API returns it
print("X range:", pos[:, 0].min(), pos[:, 0].max())
print("Y range:", pos[:, 1].min(), pos[:, 1].max())

# Plot UE positions vs TX positions
plt.figure(figsize=(10, 8))
plt.scatter(pos[:, 0], pos[:, 1], s=5, alpha=0.5, label='UEs')
plt.scatter([8.5, 280, -292], [21, 275, 295], c='red', s=100, marker='^', label='TX')
for i, (x, y, name) in enumerate([(8.5,21,'tx1'),(280,275,'tx2'),(-292,295,'tx3')]):
    plt.annotate(f'Cell {i}\n{name}', (x, y), textcoords="offset points", xytext=(10,5))
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.legend()
plt.grid()
plt.title('UE positions vs TX positions')
plt.savefig('ue_positions.png')
plt.show()
'''

'\n# After one batch of sample_positions, print the XY distribution:\nsampled_pos = cm.sample_positions(200, min_val_db=-100.0, min_dist=20.0)\npos = np.array(sampled_pos[0][0], dtype=np.float32)  # or however your API returns it\nprint("X range:", pos[:, 0].min(), pos[:, 0].max())\nprint("Y range:", pos[:, 1].min(), pos[:, 1].max())\n\n# Plot UE positions vs TX positions\nplt.figure(figsize=(10, 8))\nplt.scatter(pos[:, 0], pos[:, 1], s=5, alpha=0.5, label=\'UEs\')\nplt.scatter([8.5, 280, -292], [21, 275, 295], c=\'red\', s=100, marker=\'^\', label=\'TX\')\nfor i, (x, y, name) in enumerate([(8.5,21,\'tx1\'),(280,275,\'tx2\'),(-292,295,\'tx3\')]):\n    plt.annotate(f\'Cell {i}\n{name}\', (x, y), textcoords="offset points", xytext=(10,5))\nplt.xlabel(\'X (m)\')\nplt.ylabel(\'Y (m)\')\nplt.legend()\nplt.grid()\nplt.title(\'UE positions vs TX positions\')\nplt.savefig(\'ue_positions.png\')\nplt.show()\n'

## 10 · Signal Processing Functions

In [13]:
# ── codebook shape: [C, Lmax, Nt, 1] — matches author's spare_peel_channels ──
def compute_rsrp(H, codebook):
    # H: [C, U, Nr, Nt], codebook: [C, Lmax, Nt, 1]
    C_local, U, Nr_local, Nt_local = H.shape
    _, Lmax_local, _, _ = codebook.shape          # ← unpack 4 dims now
    rsrp_all = np.zeros((C_local, U, Lmax_local), dtype=np.float32)
    for c in range(C_local):
        for l in range(Lmax_local):
            bf_signal = H[c] @ codebook[c, l, :, 0:1]   # [U, Nr, Nt] @ [Nt, 1] → [U, Nr, 1]
            rsrp_all[c, :, l] = np.real(
                np.sum(np.abs(bf_signal) ** 2, axis=(-1, -2))   # sum over Nr and 1
            ) / Nt_local
    rsrp_flat     = rsrp_all.reshape(U, -1)
    best_flat     = np.argmax(rsrp_flat, axis=-1)
    best_cell_idx = (best_flat // Lmax_local).astype(np.int16)
    best_beam_idx = (best_flat  % Lmax_local).astype(np.int16)
    best_rsrp     = rsrp_flat[np.arange(U), best_flat].astype(np.float32)
    return best_rsrp, best_beam_idx, best_cell_idx


def beamformer_to_beamspace(f):
    # f: [Nt] or [Nt, 1]
    f = np.array(f).flatten()    # always squeeze to [Nt]
    F_2d   = f.reshape(Nx, Ny)
    bs_map = U_NxNx.conj().T @ F_2d @ U_NyNy
    return bs_map.astype(ctype)


def build_beamspace_observation(H, codebook):
    # H: [C, U, Nr, Nt], codebook: [C, Lmax, Nt, 1]
    C_local, U, _, _ = H.shape
    _, Lmax_local, _, _ = codebook.shape           # ← 4 dims
    rsrp, ssbri, bs_assign = compute_rsrp(H, codebook)
    rsrp_db   = 10.0 * np.log10(rsrp + 1e-10)
    rsrp_min, rsrp_max = rsrp_db.min(), rsrp_db.max()
    rsrp_norm = (rsrp_db - rsrp_min) / (rsrp_max - rsrp_min + 1e-10)
    active_regions = np.zeros((C_local, Lmax_local, 2, Nx1, Ny1), dtype=np.float32)
    for c in range(C_local):
        for l in range(Lmax_local):
            bs_map = beamformer_to_beamspace(codebook[c, l, :, 0])  # squeeze trailing 1
            active_regions[c, l, 0] = bs_map.real
            active_regions[c, l, 1] = bs_map.imag
    norms = np.linalg.norm(
        np.abs(active_regions[:, :, 0] + 1j * active_regions[:, :, 1]),
        axis=(2, 3), keepdims=True
    )
    active_regions[:, :, 0] /= (norms + 1e-10)
    active_regions[:, :, 1] /= (norms + 1e-10)
    active_regions_cat = np.concatenate(
        [active_regions[:, :, 0], active_regions[:, :, 1]], axis=1)
    rsrp_scaling = 1e-7 * np.ones((C_local, Lmax_local, 1, 1), dtype=np.float32)
    for u in range(U):
        rsrp_scaling[bs_assign[u], ssbri[u], 0, 0] += rsrp_norm[u]
    rsrp_scaling_tiled = np.tile(rsrp_scaling[:, :, 0, 0], 2)
    user_counts = np.zeros((C_local, Lmax_local, 1, 1), dtype=np.float32)
    for u in range(U):
        user_counts[bs_assign[u], ssbri[u], 0, 0] += 1.0
    user_counts_tiled = np.tile(user_counts[:, :, 0, 0], 2)
    full_obs = np.zeros((C_local, 2 * Lmax_local, Nx1 + 1, Ny1 + 1), dtype=np.float32)
    full_obs[:, :, :Nx1, :Ny1]  = active_regions_cat
    full_obs[:, :, Nx1,  Ny1-1] = rsrp_scaling_tiled
    full_obs[:, :, Nx1-1, Ny1]  = user_counts_tiled
    return full_obs

#def compute_svd_rate(H):
   # noise_power_dbm = -174.0 + 10.0 * np.log10(NOISE_BW) + NOISE_FIGURE_DB
   # sigma2          =  10.0**((noise_power_dbm - 30.0)/10.0)    ## in Watts 
  #  S         = np.linalg.svd(H, compute_uv=False)
   # svd_rates = np.sum(np.log2(1.0 + S ** 2 / sigma2), axis=-1)
   # return svd_rates.astype(np.float32)

def compute_svd_rate(H):
    # Author's implementation: sigma2=1, channel power IS the SNR
    #Channels were already scaled by 10^pathgain during generation (10^6)
    S = np.linalg.svd(H, compute_uv=False)
    svd_rates = np.sum(np.log2(1.0 + S**2), axis=-1) 
    return svd_rates.astype(np.float32)  

## 11 · Dataset Assembly

In [14]:


# ── DFT codebook shared by both build functions ───────────────────────────────
# Shape [C, Lmax, Nt, 1] — exactly matches author's spare_peel_channels default
_dft_cb_single  = DFT_codebook(n_beams=Lmax, Nx=Nx, Ny=Ny, OH=2, OV=2)  # [Lmax, Nt, 1]
DEFAULT_DFT_CODEBOOK = np.repeat(
    np.expand_dims(_dft_cb_single, 0), C, axis=0)   # [C, Lmax, Nt, 1]



def build_dataset(channel_pool, dataset_size=DATASET_SIZE_TRAIN,
                  Umin=UMIN, Umax=UMAX, new_user_prob=NEW_USER_PROB):
    from sklearn.preprocessing import MinMaxScaler
    pool_size  = len(channel_pool)
    Nr_actual  = channel_pool.shape[2]
    Nt_actual  = channel_pool.shape[3]

    beamspace_set = np.zeros((dataset_size, C, 2*Lmax, Nx1+1, Ny1+1), dtype=np.float32)
    channel_set   = np.zeros((dataset_size, C, Umax, Nr_actual, Nt_actual), dtype=ctype)
    svd_rate_set  = np.zeros((dataset_size, C, Umax), dtype=np.float32)
    n_users       = np.random.randint(Umin, Umax+1, size=dataset_size)
    log_every     = max(1, dataset_size // 20)

    print(f"Building training dataset ({dataset_size} samples)...")
    for d in range(dataset_size):
        U_d          = n_users[d]
        user_indices = np.random.choice(pool_size, size=U_d, replace=False)
        H_full       = channel_pool[user_indices].transpose(1, 0, 2, 3, 4, 5)
        H_narrow     = H_full[:, :, :, :, 0, 0]    # ← T=0, K=0 slice — matches author

        non_new_mask = ~(np.random.rand(U_d) < new_user_prob)
        if non_new_mask.sum() > 0:
            beamspace_obs = build_beamspace_observation(
                H_narrow[:, non_new_mask], DEFAULT_DFT_CODEBOOK)
        else:
            beamspace_obs = np.zeros((C, 2*Lmax, Nx1+1, Ny1+1), dtype=np.float32)

        svd_rates             = compute_svd_rate(H_narrow)
        beamspace_set[d]      = beamspace_obs
        channel_set[d, :, :U_d]  = H_narrow           # store [C, U, Nr, Nt]
        svd_rate_set[d, :, :U_d] = svd_rates
        if (d+1) % log_every == 0:
            print(f"  {d+1}/{dataset_size}")

    print("Normalizing beamspace per cell...")
    scaler_set = []
    n_fit      = min(1000, dataset_size)
    feat_size  = 2 * Lmax * (Nx1+1) * (Ny1+1)
    for c in range(C):
        scaler = MinMaxScaler()
        scaler.fit(beamspace_set[:n_fit, c].reshape(n_fit, feat_size))
        for start in range(0, dataset_size, 2000):
            end  = min(start+2000, dataset_size)
            flat = beamspace_set[start:end, c].reshape(end-start, feat_size)
            beamspace_set[start:end, c] = scaler.transform(flat).reshape(
                end-start, 2*Lmax, Nx1+1, Ny1+1)
        scaler_set.append(scaler)
        print(f"  Cell {c} normalized")

    beamspace_set = beamspace_set.reshape(
        [dataset_size, C*2*Lmax, Nx1+1, Ny1+1]).transpose(0, 2, 3, 1)
    print(f"beamspace_set : {beamspace_set.shape}")
    print(f"channel_set   : {channel_set.shape}")
    print(f"svd_rate_set  : {svd_rate_set.shape}")
    return beamspace_set, channel_set, svd_rate_set, scaler_set


def build_eval_dataset(channel_pool, train_scalers,      # ← accepts training scalers
                        dataset_size=DATASET_SIZE_EVAL,
                        Umin=UMIN, Umax=UMAX):
    pool_size  = len(channel_pool)
    T_dim      = channel_pool.shape[4]
    K_dim      = channel_pool.shape[5]
    Nr_actual  = channel_pool.shape[2]
    Nt_actual  = channel_pool.shape[3]

    beamspace_set = np.zeros((dataset_size, C, 2*Lmax, Nx1+1, Ny1+1), dtype=np.float32)
    channel_set   = np.zeros((dataset_size, C, Umax, T_dim, K_dim, Nr_actual, Nt_actual),
                              dtype=ctype)
    svd_rate_set  = np.zeros((dataset_size, C, Umax), dtype=np.float32)
    n_users       = np.random.randint(Umin, Umax+1, size=dataset_size)
    log_every     = max(1, dataset_size // 10)

    print(f"Building eval dataset ({dataset_size} samples)...")
    for d in range(dataset_size):
        U_d          = n_users[d]
        user_indices = np.random.choice(pool_size, size=U_d, replace=False)
        H_full       = channel_pool[user_indices].transpose(1, 0, 2, 3, 4, 5)
        H_narrow     = H_full[:, :, :, :, 0, 0]    # ← T=0, K=0 — matches author

        # no new_user_prob masking in eval — matches author's spare_peel_channels
        beamspace_obs = build_beamspace_observation(H_narrow, DEFAULT_DFT_CODEBOOK)

        svd_rates             = compute_svd_rate(H_narrow)
        beamspace_set[d]      = beamspace_obs
        channel_set[d, :, :U_d] = H_full.transpose(0, 1, 4, 5, 2, 3)
        svd_rate_set[d, :, :U_d] = svd_rates
        if (d+1) % log_every == 0:
            print(f"  {d+1}/{dataset_size}")

    # ── reuse training scalers — do NOT fit new ones on eval data ────────────
    feat_size = 2 * Lmax * (Nx1+1) * (Ny1+1)
    for c in range(C):
        scaler = train_scalers[c]                   # ← reuse from build_dataset
        for start in range(0, dataset_size, 500):
            end  = min(start+500, dataset_size)
            flat = beamspace_set[start:end, c].reshape(end-start, feat_size)
            beamspace_set[start:end, c] = scaler.transform(flat).reshape(
                end-start, 2*Lmax, Nx1+1, Ny1+1)

    beamspace_set = beamspace_set.reshape(
        [dataset_size, C*2*Lmax, Nx1+1, Ny1+1]).transpose(0, 2, 3, 1)
    print(f"beamspace_set : {beamspace_set.shape}")
    print(f"channel_set   : {channel_set.shape}")
    print(f"svd_rate_set  : {svd_rate_set.shape}")
    
    return beamspace_set, channel_set, svd_rate_set

Generating DFT codebook: 16 beams for 16×16 array (OH=2, OV=2)


## 12 · Save / Load Helpers

In [15]:
def save_dataset(beamspace_set, channel_set, svd_rate_set, scaler_set, filepath):
    import pickle
    os.makedirs(os.path.dirname(os.path.abspath(filepath)), exist_ok=True)
    with open(filepath, "wb") as f:
        pickle.dump([beamspace_set, channel_set, svd_rate_set, scaler_set], f)
    size_mb = os.path.getsize(filepath) / 1e6
    print(f"Saved → {filepath}  ({size_mb:.1f} MB)")


def load_dataset(filepath):
    import pickle
    with open(filepath, "rb") as f:
        beamspace_set, channel_set, svd_rate_set, scaler_set = pickle.load(f)
    print(f"Loaded {filepath}")
    print(f"  beamspace : {beamspace_set.shape}")
    print(f"  channels  : {channel_set.shape}")
    print(f"  svd_rates : {svd_rate_set.shape}")
    return beamspace_set, channel_set, svd_rate_set, scaler_set

## 13 · Generate Eval Dataset
Ray-traces UE positions and builds **only the eval dataset** (2000 samples).
Training dataset is NOT generated — we use the saved model directly.

In [16]:
'''
CHANNEL_POOL_PATH = "/teamspace/studios/this_studio/outputs/channel_pool_scene_A.npy"
TRAIN_PICKLE_PATH = "/teamspace/studios/this_studio/outputs/6pg_scene_A_20000_inputs_outputs_16_16_16_16_16.pickle"   # set this to skip dataset generation next run
EVAL_PICKLE_PATH  = "/teamspace/studios/this_studio/outputs/eval_6pg_scene_A_2000_inputs_outputs_16_16_16_16_16.pickle"   # set this to skip dataset generation next run
'''

'\nCHANNEL_POOL_PATH = "/teamspace/studios/this_studio/outputs/channel_pool_scene_A.npy"\nTRAIN_PICKLE_PATH = "/teamspace/studios/this_studio/outputs/6pg_scene_A_20000_inputs_outputs_16_16_16_16_16.pickle"   # set this to skip dataset generation next run\nEVAL_PICKLE_PATH  = "/teamspace/studios/this_studio/outputs/eval_6pg_scene_A_2000_inputs_outputs_16_16_16_16_16.pickle"   # set this to skip dataset generation next run\n'

In [17]:
CHANNEL_POOL_PATH = ""
TRAIN_PICKLE_PATH = ""
EVAL_PICKLE_PATH  = ""

# ── Step 1: Channel pool ───────────────────────────────────────────────────
if CHANNEL_POOL_PATH:
    print(f'Skipping ray-tracing — loading pool from: {CHANNEL_POOL_PATH}')
    channel_pool = np.load(CHANNEL_POOL_PATH)
    pos_path     = CHANNEL_POOL_PATH.replace('.npy', '_ue_pos.npy')
    ue_positions = np.load(pos_path) if os.path.exists(pos_path) else None
    print(f'Channel pool shape: {channel_pool.shape}')
else:
    print(f'Setting up Scene {SCENE}...')
    if SCENE == 'A':
        scene, cm = setup_scene_munich()
    elif SCENE == 'B':
        scene, cm = setup_scene_paris()
    else:
        raise ValueError(f"Unknown SCENE '{SCENE}'.")

    # ── Phase 2: real arrays for channel generation ────────────────────────
    scene.tx_array = PlanarArray(num_rows=Ny, num_cols=Nx//2,
                                  vertical_spacing=0.5, horizontal_spacing=0.5,
                                  pattern="tr38901", polarization="VH")
    scene.rx_array = PlanarArray(num_rows=Nr//2, num_cols=1,
                                  vertical_spacing=0.5, horizontal_spacing=0.5,
                                  pattern="tr38901", polarization="cross")

    print(f'Generating {UE_POOL_SIZE} UE channels...')
    channel_pool, ue_positions = generate_ue_channels(scene, cm,
                                                       dataset_size=UE_POOL_SIZE,
                                                       batch_size=RT_BATCH_SIZE)
    pool_path = os.path.join(OUTPUT_DIR, f'channel_pool_scene_{SCENE}.npy')
    pos_path  = os.path.join(OUTPUT_DIR, f'ue_positions_scene_{SCENE}.npy')
    np.save(pool_path, channel_pool)
    np.save(pos_path,  ue_positions)
    print(f'Pool saved to {pool_path}')
    print(f'Positions saved to {pos_path}')
    print(f'Channel pool shape: {channel_pool.shape}')

# ── Step 2: Training + Eval datasets ──────────────────────────────────────
if TRAIN_PICKLE_PATH and EVAL_PICKLE_PATH:
    print(f'Loading training dataset from: {TRAIN_PICKLE_PATH}')
    _, _, _, train_scalers = load_dataset(TRAIN_PICKLE_PATH)
    print(f'Loading eval dataset from: {EVAL_PICKLE_PATH}')
    bs_eval, ch_eval, svd_eval, _ = load_dataset(EVAL_PICKLE_PATH)
else:
    print('\n[1/2] Building training dataset...')
    bs_train, ch_train, svd_train, train_scalers = build_dataset(
        channel_pool, dataset_size=DATASET_SIZE_TRAIN)
    train_name = (f'{PATH_GAIN_EXP}pg_scene_{SCENE}_{DATASET_SIZE_TRAIN}_'
                  f'inputs_outputs_{Nx}_{Ny}_{Nx1}_{Ny1}_{Lmax}.pickle')
    train_path = os.path.join(OUTPUT_DIR, train_name)
    save_dataset(bs_train, ch_train, svd_train, train_scalers, train_path)
    print(f'Training dataset saved: {train_path}')
    del bs_train, ch_train, svd_train

    print('\n[2/2] Building eval dataset...')
    bs_eval, ch_eval, svd_eval = build_eval_dataset(
        channel_pool, train_scalers, dataset_size=EVAL_SIZE)
    eval_name = (f'eval_{PATH_GAIN_EXP}pg_scene_{SCENE}_{EVAL_SIZE}_'
                 f'inputs_outputs_{Nx}_{Ny}_{Nx1}_{Ny1}_{Lmax}.pickle')
    eval_path = os.path.join(OUTPUT_DIR, eval_name)
    save_dataset(bs_eval, ch_eval, svd_eval, train_scalers, eval_path)
    print(f'Eval dataset saved: {eval_path}')

print(f'\nEval dataset ready:')
print(f'  beamspace : {bs_eval.shape}')
print(f'  channels  : {ch_eval.shape}')
print(f'  svd_rates : {svd_eval.shape}')

Setting up Scene A...
ISD = 450.3m
Generating 1000 UE channels...
  Array config: Nr=4, Nt=256, C=3
  Generating channels 0–80 / 1000


2026-05-21 21:58:02.899179: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


  Generating channels 80–160 / 1000
  Generating channels 160–240 / 1000
  Generating channels 240–320 / 1000
  Generating channels 320–400 / 1000
  Generating channels 400–480 / 1000
  Generating channels 480–560 / 1000
  Generating channels 560–640 / 1000
  Generating channels 640–720 / 1000
  Generating channels 720–800 / 1000
  Generating channels 800–880 / 1000
  Generating channels 880–960 / 1000
  Generating channels 960–1000 / 1000
  Retrying 28 failed positions...
  Done. channel_pool shape: (1000, 3, 4, 256, 1, 20)
Pool saved to /teamspace/studios/this_studio/outputs/channel_pool_scene_A.npy
Positions saved to /teamspace/studios/this_studio/outputs/ue_positions_scene_A.npy
Channel pool shape: (1000, 3, 4, 256, 1, 20)

[1/2] Building training dataset...
Building training dataset (20000 samples)...
  1000/20000
  2000/20000
  3000/20000
  4000/20000
  5000/20000
  6000/20000
  7000/20000
  8000/20000
  9000/20000
  10000/20000
  11000/20000
  12000/20000
  13000/20000
  14000/2

## 14 · Communication Functions

In [18]:
'''
scene, cm = setup_scene_munich()
'''

'\nscene, cm = setup_scene_munich()\n'

In [19]:
print("Channel pool path used:", CHANNEL_POOL_PATH)
print("Channel pool shape:", channel_pool.shape)
print("Channel pool power per cell:")
for c in range(C):
    print(f"  Cell {c}: {np.mean(np.abs(channel_pool[:, c])**2):.4f}")

Channel pool path used: 
Channel pool shape: (1000, 3, 4, 256, 1, 20)
Channel pool power per cell:
  Cell 0: 27.8907
  Cell 1: 289.3987
  Cell 2: 39.7563


In [20]:
print("Channel pool power per cell:")
for c in range(C):
    print(f"  Cell {c}: mean={np.mean(np.abs(channel_pool[:,c])**2):.3f}, "
          f"  zeros={np.mean(np.abs(channel_pool[:,c])**2 < 0.01)*100:.1f}%")

print("\nTransmitters in scene:")
for name, tx in scene.transmitters.items():
    print(f"  {name}: pos={tx.position.numpy()}, orient={tx.orientation.numpy()}")

Channel pool power per cell:
  Cell 0: mean=27.891,   zeros=27.9%
  Cell 1: mean=289.399,   zeros=27.9%
  Cell 2: mean=39.756,   zeros=29.8%

Transmitters in scene:
  tx1: pos=[ 8.5 21.  40. ], orient=[1.25 0.   0.  ]
  tx2: pos=[280. 275.  40.], orient=[-2.3443952  0.         0.       ]
  tx3: pos=[-292.  295.   40.], orient=[-0.7971976  0.         0.       ]


In [21]:
############
## Cell #5
############

#############################
### Communication Part
#############################



"""
Neural Codebook Design for MIMO Network Beam Management
========================================================
Network Beamspace Learning (NBL) — Communications Team Functions
IMPLEMENTED

Paper: "Neural Codebook Design for MIMO Network Beam Management"
       Dreifuerst & Heath Jr., IEEE Trans. Wireless Commun., Vol. 24, No. 5, May 2025

This file provides the complete, working implementations of every function
that was previously marked as a placeholder in the AI-model scaffold.
It is designed to slot directly into the colleague's NBL notebook: the
function signatures, argument names, and return-value shapes are all
preserved exactly as declared in the placeholder stubs.

Functions implemented
─────────────────────
Training-path (differentiable):
    tf_deconvert_beamspace        – beamspace → SSB codebook vectors   (Eq.28 inverse)
    tf_deconvert_beamspace_csi    – beamspace → CSI-RS precoder matrices
    tf_rsrp_rx_comb               – SSB RSRP + best cell/beam selection  (Eq.7–9)
    tf_rsrp_rx_comb_csistep       – CSI-RS RSRP, fixed cell assignment   (Eq.13–14,19–20)
    tf_select_subset              – hierarchical CSI-RS subset selection  (Eq.12)
    tf_se_mumimo                  – MU-MIMO achievable SE                 (Eq.15–19)
    select_svd_rate               – pick per-user ideal SVD rate          (Eq.30–31)

Evaluation-only (numpy / numba, not in gradient path):
    tf_get_RZF                    – RZF digital precoder                  (Eq.23–24)
    tf_apply_chan_F               – apply selected analog precoder to H
    tf_MU_MIMO_Opt                – full MU-MIMO ESSE                     (Eq.25)

Baseline:
    DFT_codebook                  – oversampled 2-D DFT codebook

Drop this import block at the top of the colleague's Cell 5 notebook cell
in place of the `raise NotImplementedError` stubs.
"""
    ##
    ##
    ##
####  #####
 ########
   #####
    ##


# ──────────────────────────────────────────────────────────────────────────────
# Imports
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np
import numba
import tensorflow as tf
from tensorflow import keras
from functools import partial
import pickle
import os
import gc

# ──────────────────────────────────────────────────────────────────────────────
# SYSTEM CONSTANTS  (paper Section V)
# ──────────────────────────────────────────────────────────────────────────────
pi   = np.pi
ctype = np.complex64

# Array geometry
Nx   = 16          # Horizontal antenna elements
Ny   = 16          # Vertical antenna elements
Nt   = Nx * Ny     # Total transmit antennas = 256
Nr   = 4           # UE receive antennas
C    = 3           # Number of cells / base stations

# Beamspace sampling dimensions  (must satisfy Nx1 >= Nx, Ny1 >= Ny, Eq.28)
Nx1  = 16
Ny1  = 16

# Angular grid used to build the steering-vector matrix U  (Eq.26)
theta_steps = Nx1
phi_steps   = Ny1
theta_min = 3  * pi / 24
theta_max = 21 * pi / 24
phi_min   = 3  * pi / 24
phi_max   = 21 * pi / 24
thetas = np.linspace(theta_min, theta_max, num=theta_steps)
phis   = np.linspace(phi_min,   phi_max,   num=phi_steps)

# Codebook sizes
Lmax  = 16         # SSB codebook size
NCSI  = 16         # CSI-RS subset size per cell
N_csi = 8          # CSI-RS beams per SSB beam
Bg    = 4          # Beam-group / pilot-port size

# Training
BATCH_SIZE = 64
UMAX       = 20
UMIN       = 8

# Channel / noise
subcarrier_spacing = 30e3 * 12   # Hz  (RB spacing = 30 kHz × 12)
Ktotal             = 20           # OFDM symbols used in RSRP averaging
noise_figure_dB    = 7.0


# ──────────────────────────────────────────────────────────────────────────────
# ARRAY RESPONSE HELPER
# ──────────────────────────────────────────────────────────────────────────────

def array_resp(theta, N, lambda_c=1.0, d=None):
    """
    ULA array response vector for direction theta with N elements.

    a_N(theta) = (1/sqrt(N)) * [1, e^{j pi cos(theta)}, ..., e^{j pi (N-1) cos(theta)}]

    Returns shape [1, N]  (row vector, same convention as original notebook).
    """
    if d is None:
        d = lambda_c / 2.0
    n = np.expand_dims(np.arange(N), axis=1)                             # [N, 1]
    vander = np.exp(1j * pi * 2 * n * d * np.cos(theta) / lambda_c).astype(ctype).T
    return vander   # [1, N]


# ──────────────────────────────────────────────────────────────────────────────
# BEAMSPACE TRANSFORMATION MATRICES  (paper Eq.26–28)
#
# Original notebook builds these from the angular grid (thetas / phis).
# The same matrices are used for both the forward (codebook → beamspace)
# and inverse (beamspace → codebook) transforms throughout.
# ──────────────────────────────────────────────────────────────────────────────

# U_NyNy  shape [Ny1, Ny]  — row k is a_Ny(phis[k])
U_NyNy = np.array(
    [array_resp(phis[i], Ny) for i in range(Ny1)]
).astype(ctype)[:, 0, :]                               # [Ny1, Ny]

# U_NxNx  shape [Nx, Nx1]  — column k is a_Nx(thetas[k])
U_NxNx = np.array(
    [array_resp(thetas[i], Nx) for i in range(Nx1)]
).astype(ctype)[:, 0, :].T                             # [Nx, Nx1]

# Pseudo-inverses used in the beamspace → codebook direction
U_NyNy_inv = np.linalg.pinv(U_NyNy)                   # [Ny, Ny1]
U_NxNx_inv = np.linalg.pinv(U_NxNx)                   # [Nx1, Nx]

# Frobenius-normalise all four matrices (matches original notebook exactly)
U_NyNy     = U_NyNy     / np.linalg.norm(U_NyNy,     'fro')
U_NxNx     = U_NxNx     / np.linalg.norm(U_NxNx,     'fro')
U_NyNy_inv = U_NyNy_inv / np.linalg.norm(U_NyNy_inv, 'fro')
U_NxNx_inv = U_NxNx_inv / np.linalg.norm(U_NxNx_inv, 'fro')

# TensorFlow constants (cast once, reused inside @tf.function bodies)
tf_U_NyNy     = tf.constant(U_NyNy,     dtype=tf.complex64)   # [Ny1, Ny]
tf_U_NxNx     = tf.constant(U_NxNx,     dtype=tf.complex64)   # [Nx, Nx1]
tf_U_NyNy_inv = tf.constant(U_NyNy_inv, dtype=tf.complex64)   # [Ny, Ny1]
tf_U_NxNx_inv = tf.constant(U_NxNx_inv, dtype=tf.complex64)   # [Nx1, Nx]

# Eye matrices used inside MU-MIMO rate computations
tf_nr_eye      = tf.eye(Nr, dtype=tf.complex64)                # [Nr, Nr]
tf_nr_eye_U    = tf.repeat(
    tf.reshape(tf.eye(Nr, dtype=tf.complex64), [1, Nr, Nr]),
    [UMAX], axis=0
)                                                              # [Umax, Nr, Nr]


# ──────────────────────────────────────────────────────────────────────────────
# LOW-LEVEL MATH UTILITIES
# ──────────────────────────────────────────────────────────────────────────────

@tf.function()
def tflog2(x):
    """Numerically stable log base-2."""
    return tf.math.log(x) / tf.math.log(tf.constant(2, dtype=x.dtype))


@tf.function()
def tf_10log10(x):
    """10 * log10(x) in TensorFlow."""
    return 10.0 * tf.math.log(x + 1e-15) / tf.math.log(tf.constant(10.0, dtype=x.dtype))


@tf.function()
def tf_pinv(a, rcond=None):
    """
    Complex-valued pseudo-inverse via SVD.

    TensorFlow's built-in tf.linalg.pinv does not handle complex64 on all
    devices; this implementation follows the reference from the original
    notebook and is safe for batched complex tensors.
    """
    dtype = a.dtype.as_numpy_dtype

    if rcond is None:
        def _dim(d):
            v = a.shape[d]
            return v if v is not None else tf.shape(a)[d]
        nr, nc = _dim(-2), _dim(-1)
        if isinstance(nr, int) and isinstance(nc, int):
            max_rc = float(max(nr, nc))
        else:
            max_rc = tf.cast(tf.maximum(nr, nc), dtype)
        rcond = 10.0 * max_rc * np.finfo(dtype).eps

    rcond = tf.convert_to_tensor(rcond, dtype=dtype)

    s, u, v = tf.linalg.svd(a, full_matrices=False, compute_uv=True)

    # Saturate small singular values to inf  →  1/s = 0 without NaN gradients
    cutoff = tf.cast(rcond, s.dtype) * tf.reduce_max(s, axis=-1)
    s = tf.where(s > cutoff[..., None], s, np.array(np.inf, dtype))

    # pinv = V * diag(1/s) * U^H
    a_pinv = tf.matmul(
        v / tf.cast(s[..., None, :], dtype=dtype),
        u, adjoint_b=True
    )
    if a.shape is not None and a.shape.rank is not None:
        a_pinv.set_shape(a.shape[:-2].concatenate([a.shape[-1], a.shape[-2]]))
    return a_pinv


# ──────────────────────────────────────────────────────────────────────────────
# DFT CODEBOOK BASELINE
# ──────────────────────────────────────────────────────────────────────────────

def gen_codebook(Nx, Ny, OH=1, OV=1):
    """
    Generate a full oversampled 2-D DFT codebook.

    Args:
        Nx, Ny : physical array dimensions
        OH, OV : horizontal / vertical oversampling factors

    Returns:
        codebook : shape [Nx*OH*Ny*OV, Nt]  complex64
    """
    Nt = Nx * Ny
    DFT_y = np.fft.fft(np.eye(Ny * OV))[:, :Ny] / np.sqrt(Ny)    # [Ny*OV, Ny]
    DFT_x = np.fft.fft(np.eye(Nx * OH))[:Nx, :]  / np.sqrt(Nx)   # [Nx, Nx*OH]

    all_cb = np.zeros((Nx * OH * Ny * OV, Nx, Ny), dtype=ctype)
    for i in range(OH * Nx):
        for j in range(OV * Ny):
            all_cb[i * OV * Ny + j, :, :] = DFT_x[:, i:i+1] @ DFT_y[j:j+1, :]
    return all_cb.reshape(Nx * OH * Ny * OV, Nt).astype(ctype)


def DFT_codebook(n_beams=8, Nx=Nx, Ny=Ny, OH=2, OV=2, **kwargs):
    """
    Generate an n_beams-entry DFT codebook for an Nx×Ny planar array.

    When n_beams >= Nx*Ny*OH*OV the full oversampled DFT grid is used and
    then down-selected.  For smaller codebooks a centred sub-array DFT is
    used so the beams cover the most relevant angular region.

    Args:
        n_beams   : number of beamformers to return
        Nx, Ny    : physical array dimensions
        OH, OV    : oversampling (default 2×2, matching paper Section V)

    Returns:
        codebook  : shape [n_beams, Nt, 1]  complex64
                    (last dim kept for legacy callers that expect the [Nt,1] shape)
    """
    Nt = Nx * Ny
    print(f"Generating DFT codebook: {n_beams} beams for {Nx}×{Ny} array "
          f"(OH={OH}, OV={OV})")

    if n_beams >= Nx * Ny * OH * OV:
        # Full oversampled DFT — take the first n_beams rows
        cb = gen_codebook(Nx=Nx, Ny=Ny, OH=OH, OV=OV)[:n_beams]   # [n_beams, Nt]
        cb = cb.reshape(n_beams, Nt, 1)
    else:
        # Centred sub-array DFT for compact codebooks
        cb = np.zeros((n_beams, Nx, Ny), dtype=ctype)
        if n_beams <= 8:
            Nx1_ = 2
        elif n_beams <= 32:
            Nx1_ = 4
        else:
            Nx1_ = 8
        Ny1_ = n_beams // Nx1_

        DFT_x = np.fft.fft(np.eye(Nx1_)) / np.sqrt(Nx1_)
        DFT_y = np.fft.fft(np.eye(Ny1_)) / np.sqrt(Ny1_)

        x_lo = Nx // 2 - Nx1_ // 2
        y_lo = Ny // 2 - Ny1_ // 2
        for nx in range(Nx1_):
            for ny in range(Ny1_):
                cb[nx * Ny1_ + ny,
                   x_lo:x_lo + Nx1_,
                   y_lo:y_lo + Ny1_] = DFT_x[:, nx:nx+1] @ DFT_y[ny:ny+1, :]
        cb = cb.reshape(n_beams, Nt, 1)

    # Normalise each beam to unit-power × sqrt(Nt)  (matches paper notation)
    cb = cb / (np.linalg.norm(cb, axis=1, keepdims=True) + 1e-10) * np.sqrt(Nt)
    return cb


# ──────────────────────────────────────────────────────────────────────────────
# tf_deconvert_beamspace  (paper Eq.28 inverse, SSB branch)
# ──────────────────────────────────────────────────────────────────────────────

@tf.function()
def tf_deconvert_beamspace(beamspace, Lmax=Lmax):
    """
    Convert predicted beamspace back to complex SSB codebook vectors.

    The NN outputs real and imaginary parts stacked along the last axis:
        beamspace[..., :Lmax]  → real part
        beamspace[..., Lmax:]  → imaginary part

    Inversion of Eq.28:
        complex_bs = bs_real + j * bs_imag          (re-form complex tensor)
        F[x, y]    = U_NxNx_inv[n, x] * bs[x, n, y, c, l] * U_NyNy_inv[y, n]
                                                     (two sequential einsum products)
        F_flat     = reshape + transpose → [B, C, Lmax, Nt]
        F_norm     = F / ||F||_2                     (unit-norm phase-shifter constraint)

    Args:
        beamspace : NN output, shape [B, Nx1, Ny1, C, Lmax*2]  float32
        Lmax      : SSB codebook size

    Returns:
        ssb_codebook : unit-norm complex beamformers, shape [B, C, Lmax, Nt]  complex64
    """
    # ① Reconstruct complex beamspace from stacked real/imag channels
    # beamspace shape: [B, Nx1, Ny1, C, 2*Lmax]
    bs_real = tf.cast(beamspace[..., :Lmax], tf.complex64)
    bs_imag = tf.cast(beamspace[..., Lmax:], tf.complex64)
    complex_bs = bs_real + 1j * bs_imag                          # [B, Nx1, Ny1, C, Lmax]

    # ② Apply inverse beamspace matrices (Eq.28 inverse)
    #    Step 1:  yn,bxncl -> bxycl   (vertical / phi direction)
    #    tf_U_NyNy_inv shape [Ny, Ny1], summing over n=Ny1
    tmp  = tf.einsum('yn, bxncl -> bxycl', tf_U_NyNy_inv, complex_bs)  # [B, Nx1, Ny, C, Lmax]

    #    Step 2:  nx,bnycl -> bxycl   (horizontal / theta direction)
    #    tf_U_NxNx_inv shape [Nx1, Nx], summing over n=Nx1
    beams = tf.einsum('nx, bnycl -> bxycl', tf_U_NxNx_inv, tmp)         # [B, Nx, Ny, C, Lmax]

    # ③ Reshape to [B, C, Lmax, Nt]
    #    Transpose: [B, Nx, Ny, C, Lmax] → [B, C, Lmax, Nx, Ny]
    beams = tf.transpose(beams, [0, 3, 4, 1, 2])                         # [B, C, Lmax, Nx, Ny]
    beams = tf.reshape(beams, [-1, C, Lmax, Nt])                         # [B, C, Lmax, Nt]

    # ④ Unit-norm normalisation (each beamformer has ||f||₂ = 1)
    beams = beams / (tf.norm(beams, axis=-1, keepdims=True) + 1e-8)

    return beams   # [B, C, Lmax, Nt]  complex64


# ──────────────────────────────────────────────────────────────────────────────
# tf_deconvert_beamspace_csi  (paper Eq.28 inverse, CSI-RS branch)
# ──────────────────────────────────────────────────────────────────────────────

@tf.function()
def tf_deconvert_beamspace_csi(beamspace, Lmax=Lmax, N=N_csi, bg=Bg):
    """
    Convert predicted beamspace to CSI-RS precoder matrices.

    Identical beamspace inversion to the SSB branch but the output carries
    extra dimensions for beam groups (bg) and beams-per-SSB-beam (N),
    matching the CSI-RS transmission model of Eq.13–14.

    Args:
        beamspace    : NN output, shape [B, Nx1, Ny1, bg, N, C, Lmax*2]  float32
        Lmax         : SSB codebook size
        N            : CSI-RS beams per SSB beam
        bg           : beam-group size (pilot ports per CSI-RS slot)

    Returns:
        csi_codebook : shape [B, C, Lmax, N, bg, Nt]  complex64
    """
    # ① Reconstruct complex beamspace
    # beamspace shape: [B, Nx1, Ny1, bg, N, C, 2*Lmax]
    bs_real = tf.cast(beamspace[..., :Lmax], tf.complex64)
    bs_imag = tf.cast(beamspace[..., Lmax:], tf.complex64)
    complex_bs = bs_real + 1j * bs_imag                    # [B, Nx1, Ny1, bg, N, C, Lmax]

    # ② Apply inverse beamspace matrices
    #    Step 1: phi direction —  yn, bxngscl -> bxygscl
    tmp   = tf.einsum('yn, bxngscl -> bxygscl', tf_U_NyNy_inv, complex_bs)   # [B, Nx1, Ny, bg, N, C, Lmax]

    #    Step 2: theta direction — nx, bnygscl -> bxygscl
    beams = tf.einsum('nx, bnygscl -> bxygscl', tf_U_NxNx_inv, tmp)           # [B, Nx, Ny, bg, N, C, Lmax]

    # ③ Reshape to [B, C, Lmax, N, bg, Nt]
    #    Current layout: [B, Nx, Ny, bg, N, C, Lmax]
    #    Target axes:    [ b, c,  l,  N, g, Nx, Ny ] → [B, C, Lmax, N, bg, Nt]
    beams = tf.transpose(beams, [0, 5, 6, 4, 3, 1, 2])    # [B, C, Lmax, N, bg, Nx, Ny]
    beams = tf.reshape(beams, [-1, C, Lmax, N, bg, Nt])    # [B, C, Lmax, N, bg, Nt]

    # ④ Unit-norm per beamformer
    beams = beams / (tf.norm(beams, axis=-1, keepdims=True) + 1e-8)

    return beams   # [B, C, Lmax, N, bg, Nt]  complex64


# ──────────────────────────────────────────────────────────────────────────────
# tf_rsrp_rx_comb  (paper Eq.7–9 — SSB stage)
# ──────────────────────────────────────────────────────────────────────────────

@tf.function()
def tf_rsrp_rx_comb(H, beams):
    """
    SSB RSRP computation with digital combining + best cell/beam selection.

    Implements:
        RSRP_{c,i,u} = ||H_{c,u} f_{c,i}||²_F          (Eq.7, summed over Nr)
        (b̂_u, m̂_u) = argmax_{c,i} RSRP_{c,i,u}        (Eq.9)

    Design notes for backprop:
      • phat stays in the TF computation graph — gradients flow through ||H f||²
        back to the beamformer values f, and from there to the NN weights.
      • mhat and bhat are discrete integers; the caller should treat them as
        stop_gradient'd indices (no differentiable path through argmax).

    Args:
        H     : channel tensor,  shape [B, C, U, Nr, Nt]  complex64
        beams : SSB codebook,    shape [B, C, Lmax, Nt]   complex64

    Returns:
        phat  : best RSRP per user,         shape [B, U]   float32
        mhat  : best SSB beam index,        shape [B, U]   int32
        bhat  : best cell-assignment index, shape [B, U]   int32
    """
    B, C_, U, Nr_, Nt_ = H.shape
    _, _, Lmax_, _     = beams.shape

    # Compute received power for every (batch, user, cell, beam) combination:
    #   einsum 'bcurt, bclt -> bucrl'
    #     H     : [B, C, U, Nr, Nt]
    #     beams : [B, C, Lmax, Nt]
    #     out   : [B, U, C, Nr, Lmax]  — Nt contracted
    # Then ||·||₂ over the Nr dimension → RSRP [B, U, C, Lmax]
    rsrps = tf.abs(
        tf.norm(
            tf.einsum('bcurt, bclt -> bucrl', H, beams),
            axis=-2                                                # reduce Nr
        )
    )                                                              # [B, U, C, Lmax]

    # Flatten (C, Lmax) → single index to use argmax
    flat_rsrps = tf.reshape(rsrps, [B, U, -1])                   # [B, U, C*Lmax]
    phat       = tf.reduce_max(flat_rsrps, axis=-1)               # [B, U]
    beam_alloc = tf.argmax(flat_rsrps, axis=-1)                   # [B, U]  int64

    # Recover separate cell and beam indices from the flat index
    # flat_idx = c * Lmax + l  →  c = flat_idx // Lmax, l = flat_idx % Lmax
    i1   = beam_alloc // Lmax_
    i2   = beam_alloc - i1 * Lmax_
    bhat = tf.cast(i1, tf.int32)   # cell assignment
    mhat = tf.cast(i2, tf.int32)   # SSB beam index

    return phat, mhat, bhat


# ──────────────────────────────────────────────────────────────────────────────
# tf_rsrp_rx_comb_csistep  (paper Eq.13–14, 19–20 — CSI-RS stage)
# ──────────────────────────────────────────────────────────────────────────────

@tf.function()
def _select_bu(args):
    """Gather RSRP of cell b for one (batch*user) element."""
    flat_rsrp, b_flat = args
    return flat_rsrp[b_flat]


@tf.function()
def tf_rsrp_rx_comb_csistep(H, beams, b):
    """
    CSI-RS RSRP computation with cell assignment b fixed from the SSB stage.

    Users only search over the Ncsi precoders of their assigned cell; the
    cell assignment itself is not updated here.

    Implements:
        SE^{CSI-RS_i}_u  (Eq.19 per-layer SINR sum) reduced to a power metric
        î_{c,u} = argmax_i SE^{CSI-RS_i}_u   (Eq.20)

    Note: the full LMMSE-SINR version is in tf_se_mumimo.  This function
    uses a simpler ‖H f‖ power proxy for the subset-selection decision,
    which matches the notebook's train_step logic.

    Args:
        H     : channel tensor,    shape [B, C, U, Nr, Nt]     complex64
        beams : CSI-RS subset,     shape [B, C, Ncsi, bg, Nt]  complex64
        b     : cell assignments,  shape [B, U]                 int32

    Returns:
        phat  : best CSI-RS power per user, shape [B, U]  float32
        ihat  : best CSI-RS beam index,     shape [B, U]  int32
        bhat  : cell assignment (= b),      shape [B, U]  int32
    """
    B, C_, U, Nr_, Nt_  = H.shape
    _, _, Ncsi, bg_, _  = beams.shape

    # Power for every (batch, user, cell, precoder) combination:
    #   einsum 'bcurt, bclgt -> buclrg'
    #     H     : [B, C, U, Nr, Nt]
    #     beams : [B, C, Ncsi, bg, Nt]     (l indexes Ncsi, g indexes bg)
    #     out   : [B, U, C, Ncsi, Nr, bg]
    # ||·||₂ over Nr and bg → [B, U, C, Ncsi]
    rsrps = tf.abs(
        tf.norm(
            tf.einsum('bcurt, bclgt -> buclrg', H, beams),
            axis=[-2, -1]                                          # reduce Nr and bg
        )
    )                                                              # [B, U, C, Ncsi]

    # Restrict to the cell assigned by SSB (b is fixed)
    rsrps_flat   = tf.reshape(rsrps, [B * U, C_, Ncsi])           # [B*U, C, Ncsi]
    b_flat       = tf.reshape(b, [-1])                             # [B*U]
    rsrps_bactive = tf.vectorized_map(_select_bu, (rsrps_flat, b_flat))   # [B*U, Ncsi]
    rsrps_bactive = tf.reshape(rsrps_bactive, [B, U, Ncsi])       # [B, U, Ncsi]

    phat = tf.reduce_max(rsrps_bactive, axis=-1)                   # [B, U]
    ihat = tf.cast(tf.argmax(rsrps_bactive, axis=-1), tf.int32)   # [B, U]

    return phat, ihat, b


# ──────────────────────────────────────────────────────────────────────────────
# tf_select_subset  (paper Eq.12 — hierarchical CSI-RS subset selection)
# ──────────────────────────────────────────────────────────────────────────────
'''
def map_fn(fn, elems, fn_output_signature):
    batch_size = tf.shape(tf.nest.flatten(elems)[0])[0]
    arr = tf.TensorArray(
        tf.complex64, size=batch_size, element_shape=fn_output_signature.shape)
    for i in tf.range(batch_size):
        arr = arr.write(i, fn(tf.nest.map_structure(lambda x: x[i], elems)))
    return arr.stack()




@tf.function()
def sel_csi_num(SSBRI, Ncsi=32, Lmax=Lmax, N=8.0):
    # select the strongest n_cri number based on which beams are reported
    counts = tf.math.bincount(SSBRI, minlength=Lmax, maxlength=Lmax)
    # split allocation based on number selected relative to Lmax
    number_per_beam = tf.round(tf.cast(counts, tf.float32) / tf.reduce_max([tf.cast(tf.reduce_sum(counts), tf.float32), 1.0]) * Ncsi)
    number_per_beam = tf.clip_by_value(number_per_beam, 0.0, N)
    missed = tf.expand_dims(Ncsi - tf.reduce_sum(number_per_beam), axis=0)
    first_unmissed = tf.reshape(tf.argmin(number_per_beam), [1, 1])
    unmissed_val = tf.reduce_min(number_per_beam)
    if missed > 0:
        if missed > N:
            number_per_beam = tf.tensor_scatter_nd_add(number_per_beam, first_unmissed, [N])
            number_per_beam = tf.tensor_scatter_nd_add(number_per_beam, first_unmissed+1, missed-[N])
        else:
            number_per_beam = tf.tensor_scatter_nd_add(number_per_beam, first_unmissed, missed)
    if missed < 0:
        first_unmissed = tf.reshape(tf.argmax(number_per_beam), [1, 1])
        max_beams = tf.expand_dims(tf.reduce_max(number_per_beam), axis=0)
        if max_beams[0] < -1*missed[0]:
            missed = missed + max_beams
            number_per_beam = tf.tensor_scatter_nd_add(number_per_beam, first_unmissed, -1*max_beams)
            first_unmissed = tf.reshape(tf.argmax(number_per_beam), [1, 1])
        number_per_beam = tf.tensor_scatter_nd_add(number_per_beam, first_unmissed, missed)
    return tf.cast(number_per_beam, tf.int32)


# @tf.function() # can't use with map_fn due to unary lacking
def batch_c_iter_func(correlations, csi_codebook, b, m, c=0, Ncsi=16):
    # correlations is shape [Lmax, N] and sorted
    # b_inds is shape [U]
    m_c = tf.boolean_mask(m, (b==c))
    prop_selections = sel_csi_num(m_c, Lmax=Lmax, Ncsi=Ncsi, N=8) # [Lmax]
    csi_sub_c = tf.map_fn(lambda ell:doubletobatch((correlations[ell], prop_selections[ell], csi_codebook[ell])), tf.range(Lmax), fn_output_signature=tf.RaggedTensorSpec(ragged_rank=0, dtype=tf.complex64))
    csi_sub_c = tf.reshape(csi_sub_c, [Ncsi, -1, Nt])

    return csi_sub_c # [Ncsi, Bg, Nt]


# @tf.function() # can't use with map_fn due to unary lacking
def tobebatch(args):
    m, b, correlations, csi_codebook, Ncsi = args
    csi_sub_b = tf.map_fn(lambda c: batch_c_iter_func(correlations[c], csi_codebook[c], b, m, c=c, Ncsi=Ncsi), tf.range(3), fn_output_signature=tf.complex64)
    return csi_sub_b


@tf.function()
def doubletobatch(args):
    # passing in correlations of size [N], prop_selections[ell] scalar, csi_codebook [N, Bg, Nt], do the inside of ell for loop above
    correlations, prop_selection, csi_codebook = args
    indices = correlations[0:prop_selection]
    csi_codebook_ell = tf.gather(csi_codebook, indices, axis=0) # [propsel, Bg, Nt]
    return csi_codebook_ell


# @tf.function() # can't use with map_fn due to unary lacking
def tf_select_subset(ssb_codebook, csi_codebook, m, b, Ncsi=16): # m is [B, U]
    B, C, Lmax, N, bg, Nt = csi_codebook.shape
    correlations = tf.argsort(tf.reduce_max(tf.abs(tf.einsum('bclt,bclngt->bclng', ssb_codebook, tf.math.conj(csi_codebook))), axis=-1), axis=-1, direction='DESCENDING') # [B, C, Lmax, N] find which of the Lmax-N to use based on max over bg dimension
    csi_sub = tf.map_fn(lambda batch: tobebatch((m[batch], b[batch], correlations[batch], csi_codebook[batch], Ncsi)), tf.range(batch_size), fn_output_signature=tf.complex64)
    return csi_sub
'''





def tf_select_subset(ssb_codebook, csi_codebook, m, b, Ncsi=NCSI):
    """
    Vectorised reproduction of the paper's hierarchical CSI subset selection.
    Produces identical output to the eager sel_csi_num + per-(B,C,ell) gather
    chain, but with no map_fn / no Python control flow / no boolean_mask.

    Inputs:
      ssb_codebook : [B, C, Lmax, Nt]            complex64
      csi_codebook : [B, C, Lmax, N, bg, Nt]     complex64
      m            : [B, U] int32                served SSB beam per user
      b            : [B, U] int32                served cell per user
      Ncsi         : int                         CSI beams per cell

    Returns:
      csi_sub : [B, C, Ncsi, bg, Nt] complex64
    """
    B_dyn  = tf.shape(ssb_codebook)[0]
    C_     = csi_codebook.shape[1]
    Lmax_  = csi_codebook.shape[2]
    N_     = csi_codebook.shape[3]
    bg_    = csi_codebook.shape[4]
    Nt_    = csi_codebook.shape[5]
    Ncsi_f = tf.cast(Ncsi, tf.float32)
    N_f    = tf.cast(N_,   tf.float32)

    # ── 1. correlation, sort each SSB beam descending ────────────────────────
    # author uses |.| (not |.|^2). Argsort order is identical, but we keep |.|
    # to stay bit-for-bit with the reference.
    corr = tf.reduce_max(
        tf.abs(tf.einsum('bclt,bclngt->bclng',
                         ssb_codebook, tf.math.conj(csi_codebook))),
        axis=-1)                                              # [B,C,Lmax,N]
    sorted_idx = tf.argsort(corr, axis=-1, direction='DESCENDING')

    # ── 2. PER-SAMPLE counts (matches eager bincount; do NOT batch-average) ─
    b_oh = tf.one_hot(b, depth=C_,    dtype=tf.float32)       # [B,U,C]
    m_oh = tf.one_hot(m, depth=Lmax_, dtype=tf.float32)       # [B,U,Lmax]
    counts = tf.einsum('buc,bul->bcl', b_oh, m_oh)            # [B,C,Lmax]

    # ── 3. proportional allocation, exactly like the eager round+clip ───────
    sum_c   = tf.maximum(tf.reduce_sum(counts, axis=-1, keepdims=True), 1.0)
    npb     = tf.round(counts / sum_c * Ncsi_f)               # [B,C,Lmax]
    npb     = tf.clip_by_value(npb, 0.0, N_f)

    total          = tf.reduce_sum(npb, axis=-1, keepdims=True)   # [B,C,1]
    missed         = Ncsi_f - total                                # [B,C,1]
    missed_s       = missed[..., 0]                                # [B,C]

    # ── 4a. POSITIVE branch: argmin gets min(missed,N); argmin+1 gets rest ─
    idx_min       = tf.argmin(npb, axis=-1, output_type=tf.int32)  # [B,C]
    pos_mask      = tf.cast(missed_s > 0.0,    tf.float32)         # [B,C]
    overflow_mask = tf.cast(missed_s > N_f,    tf.float32)         # [B,C]

    add_to_min  = pos_mask * (overflow_mask * N_f
                              + (1.0 - overflow_mask) * missed_s)  # [B,C]
    add_to_next = pos_mask * overflow_mask * (missed_s - N_f)      # [B,C]

    # one_hot of an out-of-range index (Lmax) returns all zeros → graceful
    delta_pos = (tf.one_hot(idx_min,     depth=Lmax_, dtype=tf.float32)
                 * add_to_min[..., None]
               + tf.one_hot(idx_min + 1, depth=Lmax_, dtype=tf.float32)
                 * add_to_next[..., None])                          # [B,C,Lmax]

    # ── 4b. NEGATIVE branch: subtract from argmax, cascade once if needed ──
    neg_mask    = tf.cast(missed_s < 0.0, tf.float32)               # [B,C]
    abs_missed  = -missed_s                                          # [B,C]

    idx_max1    = tf.argmax(npb, axis=-1, output_type=tf.int32)     # [B,C]
    max1        = tf.reduce_max(npb, axis=-1)                       # [B,C]
    cascade     = tf.cast(max1 < abs_missed, tf.float32) * neg_mask # [B,C]
    no_cascade  = neg_mask * (1.0 - tf.cast(max1 < abs_missed, tf.float32))

    # If cascading, zero out idx_max1 first, then take argmax of the rest.
    mask_imax1  = tf.one_hot(idx_max1, depth=Lmax_, dtype=tf.float32)
    npb_zeroed  = npb - mask_imax1 * max1[..., None] * cascade[..., None]
    idx_max2    = tf.argmax(npb_zeroed, axis=-1, output_type=tf.int32)

    sub_at_max1 = cascade   * max1            \
                + no_cascade * abs_missed                            # [B,C]
    sub_at_max2 = cascade   * (abs_missed - max1)                    # [B,C]

    delta_neg = -(tf.one_hot(idx_max1, depth=Lmax_, dtype=tf.float32)
                  * sub_at_max1[..., None]
                + tf.one_hot(idx_max2, depth=Lmax_, dtype=tf.float32)
                  * sub_at_max2[..., None])                          # [B,C,Lmax]

    alloc_f = npb + delta_pos + delta_neg                            # [B,C,Lmax]
    # Note: author allows alloc[idx_min] > N after positive overflow add.
    # We do NOT re-clip to preserve bit-for-bit equivalence.
    alloc   = tf.cast(alloc_f, tf.int32)                             # [B,C,Lmax]

    # ── 5. compact gather: for each output slot k ∈ [0, Ncsi) ──────────────
    #   beam_id[b,c,k] = smallest ell with cumsum(alloc)[ell] > k
    #   rank   [b,c,k] = k - cumsum_shift(alloc)[ell]
    #   csi_sub[b,c,k] = csi_codebook[b, c, beam_id, sorted_idx[b,c,beam_id,rank]]
    alloc_cum = tf.cumsum(alloc, axis=-1)                            # [B,C,Lmax]
    ks_b = tf.broadcast_to(
        tf.range(Ncsi, dtype=tf.int32)[None, None, :],
        [B_dyn, C_, Ncsi])                                           # [B,C,Ncsi]

    beam_id = tf.searchsorted(alloc_cum, ks_b, side='right')         # [B,C,Ncsi]
    beam_id = tf.minimum(beam_id, Lmax_ - 1)                          # safety

    alloc_cum_shift = tf.concat(
        [tf.zeros([B_dyn, C_, 1], dtype=tf.int32), alloc_cum[..., :-1]],
        axis=-1)                                                      # [B,C,Lmax]
    beam_start  = tf.gather(alloc_cum_shift, beam_id,
                            axis=-1, batch_dims=2)                    # [B,C,Ncsi]
    rank        = ks_b - beam_start                                   # [B,C,Ncsi]

    # sorted_idx[b,c,beam_id,rank]
    sidx_slot = tf.gather(sorted_idx, beam_id,
                          axis=2, batch_dims=2)                       # [B,C,Ncsi,N]
    n_idx     = tf.gather(sidx_slot, rank,
                          axis=-1, batch_dims=3)                      # [B,C,Ncsi]

    # csi_codebook[b,c,beam_id,n_idx,:,:]
    cb_slot = tf.gather(csi_codebook, beam_id,
                        axis=2, batch_dims=2)                         # [B,C,Ncsi,N,bg,Nt]
    csi_sub = tf.gather(cb_slot, n_idx,
                        axis=3, batch_dims=3)                         # [B,C,Ncsi,bg,Nt]
    return csi_sub
#######################################################################################################





@tf.function()
def _csi_train_stage(csi_network, beamspace, H, SVD_rates,
                     ssb_codebook_sg, mhat_sg, bhat_sg,
                     Ncsi, Lmax, N_csi, bg):
    """
    Entire CSI forward pass + GradientTape compiled as ONE graph.

    Because tf_select_subset has no @tf.function of its own, TF traces
    straight through it when compiling this function. The tape therefore
    sees the full path:
        csi_network weights → csi_codebook → csi_sub → SEhat → csi_loss
    and gradients flow correctly.  AND the whole thing runs compiled.
    """
    csi_vars = csi_network.trainable_variables
    with tf.GradientTape() as tape:
        pred_csi     = csi_network(beamspace, training=True)
        csi_codebook = tf_deconvert_beamspace_csi(pred_csi, Lmax=Lmax,
                                                  N=N_csi, bg=bg)
        csi_sub      = tf_select_subset(ssb_codebook_sg, csi_codebook,
                                        mhat_sg, bhat_sg, Ncsi)
        _, ihat, _   = tf_rsrp_rx_comb_csistep(H, csi_sub, bhat_sg)
        SEhat        = tf_se_mumimo(csi_sub, ihat, bhat_sg, H)
        SEtrue       = select_svd_rate(SVD_rates, bhat_sg)
        csi_loss     = nbl_loss(SEtrue, SEhat)
    grads = tape.gradient(csi_loss, csi_vars)
    return csi_loss, grads




# ──────────────────────────────────────────────────────────────────────────────
# tf_se_mumimo  (paper Eq.15–19 — MU-MIMO achievable SE during CSI-RS)
# ──────────────────────────────────────────────────────────────────────────────

@tf.function()
def _extract_Hfi_bs(args):
    """Extract beamformed channel for one user given scalar index."""
    H, index = args   # H is [Ncsi, C, Nr, bg], index scalar
    return H[index]


@tf.function()
def _extract_Hfi_batch(args):
    """Vectorised extraction of beamformed channel over users."""
    H, indices = args   # H is [U, Ncsi, C, Nr, bg], indices [U]
    return tf.vectorized_map(_extract_Hfi_bs, args)


@tf.function()
def _tf_se_mumimo_batch(args):
    """
    Compute per-user SE for one batch element.

    Uses the LMMSE rate formula (Eq.18–19):
        SE_u = log₂ det(I + D_u · (I + INR_u)^{-1})
    where D_u is the signal covariance from the serving cell and
    INR_u is the sum of interference covariances from other cells.

    The log2-det form is numerically equivalent to the scalar per-stream
    log2(1+SINR) used in the paper but more stable for full-rank matrices.

    Args (packed):
        HFFH  : [U, C, Nr, Nr]  complex64
        bhat  : [U]             int32
    """
    HFFH, bhat = args
    # U×Nr×Nr tiles for the identity
    U_ = tf.shape(bhat)[0]
    tf_eye_tile = tf.tile(
        tf.reshape(tf.eye(Nr, dtype=tf.complex64), [1, Nr, Nr]),
        [U_, 1, 1]
    )                                              # [U, Nr, Nr]

    D     = tf.gather(HFFH, bhat, axis=1, batch_dims=1)  # [U, Nr, Nr]  serving cell
    infer = tf.reduce_sum(HFFH, axis=1) - D              # [U, Nr, Nr]  interference

    # log₂ det(I + D · (I + INR)⁻¹)
    Cu = tflog2(
        tf.abs(
            tf.linalg.det(
                tf_eye_tile + tf.matmul(D, tf_pinv(tf_eye_tile + infer))
            )
        )
    )                                                      # [U]
    return Cu


@tf.function()
def tf_se_mumimo(csi_sub, ihat, bhat, H):
    """
    MU-MIMO achievable SE during CSI-RS transmission (paper Eq.15–19).

    Computes:
        SE_u = sum_{nr} log₂(1 + SINR_{u,nr})    (Eq.19)
    where SINR comes from the LMMSE combiner (Eq.17–18).

    This function is the primary source of gradients that flow back through
    csi_sub → csi_network weights and (via the subset indices) → ssb_network.

    Args:
        csi_sub : selected CSI-RS precoders, shape [B, C, Ncsi, bg, Nt]  complex64
        ihat    : best CSI-RS index per user, shape [B, U]                int32
        bhat    : cell assignment per user,   shape [B, U]                int32
        H       : channel,                   shape [B, C, U, Nr, Nt]     complex64

    Returns:
        SEhat : per-user achievable SE, shape [B, U]  float32
    """
    B_, C_, U_, Nr_, Nt_ = H.shape

    # ① Effective beamformed channel for every (user, precoder) combo
    #   einsum 'bcurt, bcngt -> buncrg'
    #     H       : [B, C, U, Nr, Nt]
    #     csi_sub : [B, C, Ncsi, bg, Nt]
    #     out     : [B, U, Ncsi, C, Nr, bg]
    beamformed_chan = tf.einsum('bcurt, bcngt -> buncrg', H, csi_sub)   # [B, U, Ncsi, C, Nr, bg]

    # ② Select the ihat-th precoder for each user
    #   _extract_Hfi_batch: for each (batch, user) pair pick beamformed_chan[u, ihat[b,u]]
    Hfi = tf.transpose(
        tf.vectorized_map(_extract_Hfi_batch, (beamformed_chan, ihat)),
        [0, 2, 1, 3, 4]
    )                                                               # [B, C, U, Nr, bg]

    # ③ Outer product  H_fi H_fi^H  →  signal/interference covariance
    HFFH = tf.einsum('bcurg, bcung -> bucrn', Hfi, tf.math.conj(Hfi))  # [B, U, C, Nr, Nr]

    # ④ Per-batch rate computation
    BCu = tf.vectorized_map(_tf_se_mumimo_batch, (HFFH, bhat))     # [B, U]

    return BCu   # [B, U]  float32


# ──────────────────────────────────────────────────────────────────────────────
# select_svd_rate  (paper Eq.30–31 — ideal per-user SVD rate)
# ──────────────────────────────────────────────────────────────────────────────

@tf.function()
def _sel_svd_batch_u(args):
    """Select SVD rate for one user given cell index b."""
    svd_rates, b = args    # svd_rates: [C], b: scalar int32
    return svd_rates[b]


@tf.function()
def _sel_svd_batch(args):
    """Vectorise over users within one batch."""
    return tf.vectorized_map(_sel_svd_batch_u, args)


@tf.function()
def select_svd_rate(SVD_rates, b):
    """
    Extract per-user ideal SVD rate for their assigned cell.

    Given rates for all cells (shape [B, C, U]) and user-cell assignments b
    (shape [B, U]), return the rate each user would achieve with their serving
    cell under perfect CSI and SVD precoding (Eq.30–31).

    Args:
        SVD_rates : ideal rates for every cell–user pair, shape [B, C, U]  float32
        b         : cell assignment per user,             shape [B, U]     int32

    Returns:
        SEtrue : per-user ideal rate, shape [B, U]  float32
    """
    # Transpose to [B, U, C] so inner ops iterate over users
    SVD_rates_t = tf.transpose(SVD_rates, [0, 2, 1])              # [B, U, C]
    SEtrue = tf.vectorized_map(_sel_svd_batch, (SVD_rates_t, b))   # [B, U]
    return SEtrue


# ──────────────────────────────────────────────────────────────────────────────
# tf_apply_chan_F  (evaluation only — apply selected analog precoder)
# ──────────────────────────────────────────────────────────────────────────────

def tf_apply_chan_F(csi_sub, ihat, bhat, H):
    """
    Apply the selected CSI-RS analog precoder to the full (T, K) channel.

    Used during evaluation to obtain the effective beamformed channel
    Heff = H · F^RF, which is then passed to tf_get_RZF for RZF digital
    precoding and tf_MU_MIMO_Opt for ESSE calculation.

    NOT in the training gradient path.

    Args:
        csi_sub : selected analog precoders, shape [B, C, Ncsi, bg, Nt]    complex64
        ihat    : best CSI-RS index per user, shape [B, U]                  int32
        bhat    : cell assignment per user,   shape [B, U]                  int32
        H       : full channel (T, K freq-domain), shape [B, C, U, T, K, Nr, Nt] complex64

    Returns:
        Heff : effective beamformed channel, shape [B, U, C, T, K, Nr, bg]  complex64
    """
    B_, C_, U_, T_, K_, Nr_, Nt_ = H.shape

    # ① Contract Nt to get beamformed channel for every (user, precoder) combo
    #   einsum 'bcutkry, bcngy -> bunctkrg'
    #     H       : [B, C, U, T, K, Nr, Nt]   (y = Nt)
    #     csi_sub : [B, C, Ncsi, bg, Nt]       (y = Nt, g = bg)
    #     out     : [B, U, Ncsi, C, T, K, Nr, bg]
    beamformed_chan = tf.einsum('bcutkry, bcngy -> bunctkrg', H, csi_sub)
    # shape: [B, U, Ncsi, C, T, K, Nr, bg]

    # ② Select the ihat-th precoder for each user (gather over Ncsi axis=2)
    #   ihat shape [B, U], batch_dims=2 iterates over (B, U)
    beamformed_chan_sel = tf.gather(beamformed_chan, ihat,
                                   axis=2, batch_dims=2)
    # shape: [B, U, C, T, K, Nr, bg]

    return beamformed_chan_sel


# ──────────────────────────────────────────────────────────────────────────────
# tf_get_RZF  (evaluation only — Regularised Zero-Forcing digital precoder)
# ──────────────────────────────────────────────────────────────────────────────

@numba.jit(nopython=True)
def _zf_precoder(H):
    """
    Per-resource-element RZF precoder (Eq.23).

    Args:
        H : beamformed channel, shape [U, Nr, bg]  complex64

    Returns:
        precoders : shape [U, bg, bg]  complex64

    The regularisation term is U*bg*I which matches the signal-to-leakage
    noise ratio maximisation approach described in [Heath & Lozano, 2018].
    """
    U_, Nr_, Bg_ = H.shape
    precoders = np.zeros((U_, Bg_, Bg_), dtype=np.complex64)

    # Gram matrix with regularisation: (sum_u H_u^H H_u + U*Bg*I)^{-1}
    reg = U_ * Bg_ * np.eye(Bg_, dtype=np.complex64)
    HuHu = reg.copy()
    for u in range(U_):
        HuHu = HuHu + H[u].conj().T @ H[u]
    HuHuinv = np.linalg.pinv(HuHu)

    for u in range(U_):
        precoders[u] = HuHuinv @ H[u].conj().T   # [bg, Nr] · [Nr, bg] → [bg, bg]

    return precoders


def tf_get_RZF(Heff, Np=32):
    """
    Compute the RZF digital precoder for every resource element (Eq.23–24).

    Implemented in NumPy/Numba (not TensorFlow) because RZF is evaluation-only
    and Numba's JIT gives a significant speed advantage for the nested loops.

    Args:
        Heff : effective beamformed channel, shape [B, U, C, T, K, Nr, bg]  complex64
               (output of tf_apply_chan_F)
        Np   : total RF chains (not actively used; kept for API compatibility)

    Returns:
        Fdigital : RZF precoder, shape [B, C, T, K, U, bg, bg]  complex64
    """
    # tf_apply_chan_F returns [B, U, C, T, K, Nr, bg]
    # Reorder to [B, C, T, K, U, Nr, bg] for the nested loops
    Heff_np = np.array(Heff).transpose([0, 2, 3, 4, 1, 5, 6])   # [B, C, T, K, U, Nr, bg]
    B_, C_, T_, K_, U_, Nr_, Bg_ = Heff_np.shape

    Fdigital = np.zeros((B_, C_, T_, K_, U_, Bg_, Bg_), dtype=np.complex64)

    for b in range(B_):
        for c in range(C_):
            for t in range(T_):
                for k in range(K_):
                    Fdigital[b, c, t, k] = _zf_precoder(Heff_np[b, c, t, k])

    # Normalise per resource element
    Fdigital = Fdigital / (np.linalg.norm(Fdigital, axis=(-2, -1), keepdims=True) + 1e-10)

    return Fdigital


# ──────────────────────────────────────────────────────────────────────────────
# tf_MU_MIMO_Opt  (evaluation only — full MU-MIMO ESSE, paper Eq.25)
# ──────────────────────────────────────────────────────────────────────────────

def tf_MU_MIMO_Opt(Heff, Fdigital, bhat):
    """
    Full MU-MIMO Effective Sum Spectral Efficiency after data transmission.

    Implements Eq.25:
        ESSE = sum_u sum_{t,k} sum_r log₂(1 + SINR_{u,t,k,r})

    where SINR is computed from:
        H_eff · F_dig: [B, U1, U2, C, T, K, Nr, Ns]  (HF product)
        D_u           = (H_eff F_dig)(H_eff F_dig)^H for user u, serving cell
        Interference  = sum_{u'≠u} (H_eff F_dig)^{u'} (...) from serving cell

    Note: bhat selects the serving cell for each user; the RZF precoder
    Fdigital is per-cell so interference is intra-cell only at the digital
    stage (inter-cell was already mitigated by analog codebook design).

    Args:
        Heff     : beamformed channel,   shape [B, U, C, T, K, Nr, bg]     complex64
        Fdigital : RZF precoder,         shape [B, C, T, K, U, bg, bg]     complex64
        bhat     : cell assignment,      shape [B, U]                       int32

    Returns:
        Cu : per-resource SE per user,   shape [B, T, K, U]  float32
    """
    B_, U_, C_, T_, K_, _, Bg_ = Heff.shape

    # Tile identity once for the whole batch+resource grid
    tf_eye_base = tf.reshape(tf.eye(Nr, dtype=tf.complex64), [1, 1, 1, 1, Nr, Nr])
    tf_eye_tiled = tf.tile(tf_eye_base, [B_, T_, K_, U_, 1, 1])   # [B, T, K, U, Nr, Nr]

    # ① H_eff · F_dig  (combining beamformed channel with digital precoder)
    #   'buctkrg, bctkwgs -> buwctkrs'
    #     Heff     : [B, U1, C, T, K, Nr, bg]   (r=Nr, g=bg, b=batch)
    #     Fdigital : [B, C, T, K, U2, bg, Ns]   (w=U2, s=Ns)
    #     out      : [B, U1, U2, C, T, K, Nr, Ns]
    HF = tf.einsum('buctkrg, bctkwgs -> buwctkrs',
                   Heff,
                   tf.cast(Fdigital, tf.complex64))                # [B, U, U, C, T, K, Nr, Ns]

    # ② Outer product  HF · HF^H  → covariance [B, U1, C, U2, T, K, Nr, Nr]
    HFFH = tf.einsum('buwctkrs, buwctkas -> bucwtkra',
                     HF, tf.math.conj(HF))                         # [B, U1, C, U2, T, K, Nr, Nr]

    # ③ Select serving cell for each user  (using bhat)
    #   gather over C axis (axis=2) with batch_dims=2 → [B, T, K, U1, U2, Nr, Nr]
    serving_HFFH = tf.transpose(
        tf.gather(HFFH, bhat, axis=2, batch_dims=2),
        [0, 3, 4, 1, 2, 5, 6]
    )                                                              # [B, T, K, U1, U2, Nr, Nr]

    # ④ Total interference = sum over transmitting users − self
    inter_HFFH   = tf.reduce_sum(serving_HFFH, axis=4)           # [B, T, K, U1, Nr, Nr]
    signal_HFFH  = tf.einsum('btkuura -> btkura', serving_HFFH)  # [B, T, K, U, Nr, Nr] diagonal
    inter_HFFH   = inter_HFFH - signal_HFFH                      # remove self from interference

    # ⑤ Log-det rate (Eq.25):
    #   Cu = log₂ det(I + U · D · (I + INR/U)^{-1})
    Cu = tflog2(
        tf.abs(
            tf.linalg.det(
                tf_eye_tiled
                + tf.matmul(
                    signal_HFFH * float(U_),
                    tf_pinv(tf_eye_tiled + inter_HFFH / float(U_))
                )
            )
        )
    )                                                              # [B, T, K, U]

    return Cu


# ──────────────────────────────────────────────────────────────────────────────
# LOSS FUNCTION  (paper Eq.32)
# ──────────────────────────────────────────────────────────────────────────────

@tf.function()
def nbl_loss(SEtrue, SEhat):
    """
    NBL Training Loss — MSE between ideal SVD rate and predicted achievable SE.

    Paper Eq.32:
        L(r, r̂) = (1/U) Σ_u (r_u − r̂_u)²

    Inactive users (those padded with zero when U < Umax) are masked out so
    they do not contribute to the gradient.

    Args:
        SEtrue : ideal per-user SVD rate,     shape [B, U]  float32
        SEhat  : predicted achievable SE,     shape [B, U]  float32

    Returns:
        loss : scalar float32
    """
    active_mask      = SEtrue > 1e-3
    se_true_active   = tf.boolean_mask(SEtrue, active_mask)
    se_hat_active    = tf.boolean_mask(SEhat,  active_mask)
    return tf.reduce_mean(tf.square(se_true_active - se_hat_active))

## 15 · Neural Network Architecture & NBL Model

In [22]:
############
## Cell #6
############

# =============================================================================
# NEURAL NETWORK ARCHITECTURE  (paper Section IV, Figure 3)
# =============================================================================

def _make_encoder_block(
    x: tf.Tensor,
    filters: int,
    kernel_size: tuple = (5, 5),
    pool: bool = True
) -> tf.Tensor:
    """
    One encoder stage: Conv2D -> (optional MaxPool).

    Paper says standard Conv2D with (128,128,256) filters and 2x2 max pooling.
    (Code uses SeparableConv2D for efficiency — we follow the paper here.)
    """
    x = keras.layers.Conv2D(
        filters     = filters,
        kernel_size = kernel_size,
        activation  = 'relu',
        padding     = 'same'
    )(x)
    if pool:
        x = keras.layers.MaxPool2D(pool_size=(2, 2), padding='same')(x)
    return x


def _make_decoder_block(
    x: tf.Tensor,
    filters: int,
    kernel_size: tuple = (5, 5)
) -> tf.Tensor:
    """
    One decoder stage: Conv2DTranspose (upsampling).

    Paper: "final 3 layers perform an inverse convolution (convolution-transpose)
    instead of using fully connected layers due to the large dimensional outputs."
    """
    x = keras.layers.Conv2DTranspose(
        filters     = filters,
        kernel_size = kernel_size,
        strides     = (2, 2),
        activation  = 'relu',
        padding     = 'same'
    )(x)
    return x


def build_ssb_network(
    Lmax:      int   = Lmax,
    Nx1:       int   = Nx1,
    Ny1:       int   = Ny1,
    C:         int   = C,
    batch_size: int  = BATCH_SIZE,
    depth:     int   = 3
) -> keras.Model:
    """
    SSB Codebook Convolutional Autoencoder  (paper Section IV, Figure 3).

    Architecture:
        Input  [B, Nx1+1, Ny1+1, C*2*Lmax]
            BatchNorm
            Conv2D(128, 2x2, valid)          <- entry conv, no pool
            Conv2D(128, 5x5) + MaxPool2D     <- encoder stage 1
            Conv2D(128, 5x5) + MaxPool2D     <- encoder stage 2
            Conv2D(256, 5x5) + MaxPool2D     <- encoder stage 3
            Conv2DTranspose(128, 5x5, s=2)   <- decoder stage 1
            Conv2DTranspose(128, 5x5, s=2)   <- decoder stage 2
            Conv2DTranspose(128, 5x5, s=2)   <- decoder stage 3
            Conv2D(C*Lmax*2, 5x5)            <- output projection
        Output [B, Nx1, Ny1, C, Lmax*2]     <- real and imag stacked

    The output represents the predicted beamspace for ALL C cells jointly.
    The last dim is [real_part(Lmax) | imag_part(Lmax)] per cell.

    ~2M parameters total for SSB + CSI networks (paper page 8).

    Args:
        Lmax       : SSB codebook size
        Nx1, Ny1   : beamspace spatial dimensions
        C          : number of cells
        batch_size : needed to reshape output correctly
        depth      : number of encoder/decoder layers (paper uses 3)

    Returns:
        keras.Model with input [B, Nx1+1, Ny1+1, C*2*Lmax]
                      and output [B, Nx1, Ny1, C, Lmax*2]
    """
    # Input: concatenated beamspace observations from all C cells
    # Shape: [B, Nx1+1, Ny1+1, C*2*Lmax]
    # The +1 in spatial dims holds the RSRP scaling and user-count channels (Eq.29)
    inputs = keras.Input(shape=(Nx1 + 1, Ny1 + 1, C * 2 * Lmax))

    # --- Normalisation ---
    x = keras.layers.BatchNormalization()(inputs)

    # --- Entry convolution (no pooling, valid padding to avoid spatial growth) ---
    x = keras.layers.Conv2D(
        filters     = 128,
        kernel_size = (2, 2),
        activation  = 'relu',
        padding     = 'valid'
    )(x)

    # --- Encoder: 3 stages of Conv + MaxPool ---
    # Filters follow the paper: (128, 128, 256)
    encoder_filters = [128, 128, 256]
    for f in encoder_filters:
        x = _make_encoder_block(x, filters=f, pool=True)

    # --- Decoder: 3 Conv2DTranspose stages (upsampling back to input resolution) ---
    # Paper: "3 layers perform an inverse convolution instead of fully connected layers"
    for _ in range(depth):
        x = _make_decoder_block(x, filters=128)

    # --- Output projection: map to C * Lmax * 2 channels (real + imag per cell) ---
    x = keras.layers.Conv2D(
        filters     = C * Lmax * 2,
        kernel_size = (5, 5),
        activation  = None,            # linear — beamspace values are unbounded
        padding     = 'same'
    )(x)

    # Reshape to [B, Nx1, Ny1, C, Lmax*2]
    # We slice [:Nx1, :Ny1] because the decoder may overshoot spatial dims
    outputs = keras.layers.Reshape((Nx1, Ny1, C, Lmax * 2))(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='ssb_network')
    return model


def build_csi_network(
    Lmax:       int  = Lmax,
    N:          int  = N_csi,
    bg:         int  = Bg,
    Nx1:        int  = Nx1,
    Ny1:        int  = Ny1,
    C:          int  = C,
    batch_size: int  = BATCH_SIZE,
    depth:      int  = 3
) -> keras.Model:
    """
    CSI-RS Codebook Convolutional Autoencoder  (paper Section IV, Figure 3).

    Identical architecture to the SSB network but with a different output shape.
    The CSI-RS codebook is a precoder matrix (multi-column) so the output has
    extra dimensions for beam groups (bg) and beams-per-SSB (N).

    Output shape: [B, Nx1, Ny1, bg, N, C, Lmax*2]
    After beamspace inversion -> [B, C, Lmax, N, bg, Nt]  (Eq.28 inverse)

    Args:
        Lmax, N, bg : codebook structure parameters
        Nx1, Ny1    : beamspace spatial dimensions
        C           : number of cells
        batch_size  : needed to reshape output
        depth       : encoder/decoder depth

    Returns:
        keras.Model with input  [B, Nx1+1, Ny1+1, C*2*Lmax]
                      and output [B, Nx1, Ny1, bg, N, C, Lmax*2]
    """
    # Same input as SSB — both networks see the same beamspace observation
    inputs = keras.Input(shape=(Nx1 + 1, Ny1 + 1, C * 2 * Lmax))

    x = keras.layers.BatchNormalization()(inputs)

    # Entry conv
    x = keras.layers.Conv2D(
        filters     = 128,
        kernel_size = (2, 2),
        activation  = 'relu',
        padding     = 'valid'
    )(x)

    # Encoder
    encoder_filters = [128, 128, 256]
    for f in encoder_filters:
        x = _make_encoder_block(x, filters=f, pool=True)

    # Decoder
    for _ in range(depth):
        x = _make_decoder_block(x, filters=128)

    # Output projection: bg * N * C * Lmax * 2 channels
    out_channels = bg * N * C * Lmax * 2
    x = keras.layers.Conv2D(
        filters     = out_channels,
        kernel_size = (5, 5),
        activation  = None,
        padding     = 'same'
    )(x)

    outputs = keras.layers.Reshape((Nx1, Ny1, bg, N, C, Lmax * 2))(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='csi_network')
    return model


# =============================================================================
# LOSS FUNCTION  (paper Eq.32)
# =============================================================================

@tf.function()
def nbl_loss(SEtrue: tf.Tensor, SEhat: tf.Tensor) -> tf.Tensor:
    """
    NBL Training Loss — MSE between ideal SVD rate and predicted achievable SE.

    Paper Eq.32:
        L(r, r_hat) = (1/U) * sum_u (r_u - r_hat_u)^2

    where:
        r_u     = sum_{nr} log2(1 + S^2_{u,nr} / sigma^2)    (ideal, Eq.31)
        r_hat_u = SE from Eq.19 using the predicted codebooks

    We mask out inactive users (those padded with zeros when U < Umax).

    Args:
        SEtrue : ideal per-user SVD rate,       shape [B, U]  float32
        SEhat  : predicted achievable SE,       shape [B, U]  float32

    Returns:
        loss : scalar MSE over active users
    """
    # Only compute loss for users that are actually present (non-zero rate)
    active_mask = SEtrue > 1e-3
    se_true_active = tf.boolean_mask(SEtrue, active_mask)
    se_hat_active  = tf.boolean_mask(SEhat,  active_mask)
    return tf.reduce_mean(tf.square(se_true_active - se_hat_active))


# =============================================================================
# NBL MODEL CLASS  (paper Section IV)
# =============================================================================

class NBL(keras.Model):
    """
    Network Beamspace Learning (NBL) model.

    Implements the full end-to-end pipeline from Figure 3 of the paper:

        Beamspace input
            |
            +---> SSB Conv-Autoencoder ---> beamspace inversion ---> SSB codebook
            |                                                              |
            |                                               SSB RSRP + beam/cell assignment
            |                                                              |
            +---> CSI Conv-Autoencoder ---> beamspace inversion ---> CSI-RS codebook
                                                                          |
                                                              CSI-RS subset selection (Eq.12)
                                                                          |
                                                          CSI-RS RSRP + beam selection (Eq.19-20)
                                                                          |
                                                              MU-MIMO achievable SE (Eq.15-19)
                                                                          |
                                                                   NBL Loss (Eq.32)

    Two-phase training (matching code structure):
        Phase 1 (SSB): SSB network trained with -RSRP loss to warm up
                       (optional — can skip and go straight to end-to-end)
        Phase 2 (E2E): Both networks trained jointly with SE-MSE loss (Eq.32)
                       Gradients flow through:
                           SE -> SINR -> LMMSE combiner -> csi_sub -> CSI-RS codebook -> csi_network
                           SE -> SINR -> LMMSE combiner -> csi_sub -> SSB codebook   -> ssb_network
                           (argmax indices are stop_gradient'd — straight-through)
    """

    def __init__(
        self,
        ssb_network: keras.Model = None,
        csi_network: keras.Model = None,
        Lmax:        int = Lmax,
        N_csi:       int = N_csi,
        bg:          int = Bg,
        Ncsi:        int = NCSI,
        batch_size:  int = BATCH_SIZE
    ):
        super().__init__()

        # Build or receive networks
        self.ssb_network = ssb_network if ssb_network else build_ssb_network(Lmax=Lmax, batch_size=batch_size)
        self.csi_network = csi_network if csi_network else build_csi_network(Lmax=Lmax, N=N_csi, bg=bg, batch_size=batch_size)

        # Store hyperparameters
        self.Lmax       = Lmax
        self.N_csi      = N_csi
        self.bg         = bg
        self.Ncsi       = Ncsi
        self.batch_size = batch_size

        # Metrics tracked during training
        self._loss_tracker     = keras.metrics.Mean(name='se_loss')
        self._ssb_rsrp_tracker = keras.metrics.Mean(name='ssb_rsrp_db')

    # -------------------------------------------------------------------------
    # FORWARD PASS (used during inference and inside GradientTape)
    # -------------------------------------------------------------------------

    def forward(
        self,
        beamspace:  tf.Tensor,
        H:          tf.Tensor
    ) -> tuple[tf.Tensor, tf.Tensor, tf.Tensor, tf.Tensor, tf.Tensor]:
        """
        Full forward pass through NBL (Figure 3).

        This is the core of the model. Everything here runs inside
        tf.GradientTape during training, so ALL TF ops must be
        differentiable (or explicitly stop_gradient'd where needed).

        Args:
            beamspace : input beamspace observation, shape [B, Nx1+1, Ny1+1, C*2*Lmax]
            H         : channel for all users/cells, shape [B, C, U, Nr, Nt]

        Returns:
            SEhat        : predicted achievable SE per user,    [B, U]
            phat_ssb     : RSRP from SSB stage (for monitoring),[B, U]
            mhat         : SSB beam index per user,             [B, U]
            bhat         : cell assignment per user,            [B, U]
            ihat         : CSI-RS beam index per user,          [B, U]
        """
        # ------------------------------------------------------------------
        # STEP 1: Predict SSB beamspace -> SSB codebook
        # NN output: [B, Nx1, Ny1, C, Lmax*2]  (real+imag stacked)
        # After inversion: [B, C, Lmax, Nt]    (complex, unit-norm)
        # ------------------------------------------------------------------
        pred_ssb_beamspace = self.ssb_network(beamspace, training=True)
        ssb_codebook = tf_deconvert_beamspace(
            pred_ssb_beamspace,
            Lmax=self.Lmax
        )  # [B, C, Lmax, Nt]

        # ------------------------------------------------------------------
        # STEP 2: SSB stage — RSRP computation + user-cell assignment (Eq.7-9)
        # phat carries gradient; mhat, bhat are discrete -> stop_gradient
        # ------------------------------------------------------------------
        phat_ssb, mhat, bhat = tf_rsrp_rx_comb(H, ssb_codebook)
        # mhat, bhat: stop_gradient applied INSIDE tf_rsrp_rx_comb by comm team
        # phat_ssb: stays in graph (||H f||^2 is smooth w.r.t. f)

        # ------------------------------------------------------------------
        # STEP 3: Predict CSI-RS beamspace -> CSI-RS codebook
        # NN output: [B, Nx1, Ny1, bg, N, C, Lmax*2]
        # After inversion: [B, C, Lmax, N, bg, Nt]
        # ------------------------------------------------------------------
        pred_csi_beamspace = self.csi_network(beamspace, training=True)
        csi_codebook = tf_deconvert_beamspace_csi(
            pred_csi_beamspace,
            Lmax=self.Lmax,
            N=self.N_csi,
            bg=self.bg
        )  # [B, C, Lmax, N, bg, Nt]

        # ------------------------------------------------------------------
        # STEP 4: CSI-RS subset selection (paper Eq.12 + proportional logic)
        # Select Ncsi beams per cell proportional to user distribution
        # Selection indices are stop_gradient'd; selected precoder values carry grad
        # ------------------------------------------------------------------
        csi_sub = tf_select_subset(
            ssb_codebook,
            csi_codebook,
            mhat,
            bhat,
            self.Ncsi
        )  # [B, C, Ncsi, bg, Nt]

        # ------------------------------------------------------------------
        # STEP 5: CSI-RS stage — RSRP with fixed cell assignment (Eq.13-14,19-20)
        # ihat indexes which CSI-RS precoder each user selected
        # ------------------------------------------------------------------
        _, ihat, _ = tf_rsrp_rx_comb_csistep(H, csi_sub, bhat)

        # ------------------------------------------------------------------
        # STEP 6: MU-MIMO achievable SE (Eq.15-19)
        # This is the main gradient signal: SE -> SINR -> LMMSE -> csi_sub -> NN
        # Uses tf.linalg.solve internally (not tf.linalg.inv) for stability
        # ------------------------------------------------------------------
        SEhat = tf_se_mumimo(csi_sub, ihat, bhat, H)  # [B, U]

        return SEhat, phat_ssb, mhat, bhat, ihat

    # -------------------------------------------------------------------------
    # TRAINING STEP  (paper Section IV, end-to-end learning)
    # -------------------------------------------------------------------------

    def train_step(self, inputs: tuple) -> dict:
        """
        One training step: forward pass + loss + backprop.

        Gradient flow (Design Aspect 1 in paper):
            Loss (Eq.32 MSE)
              -> SE (Eq.19)
              -> SINR (Eq.18) — diff through LMMSE via tf.linalg.solve
              -> csi_sub (selected precoders, values carry grad, indices don't)
              -> csi_codebook (Eq.28 inverse, matrix multiply)
              -> csi_network weights  ✓
              -> ssb_codebook (via subset selection values)
              -> ssb_network weights  ✓

        The SSB and CSI networks are trained JOINTLY in one tape.
        Separate optimizers are used (lr_ssb=4e-4, lr_csi=2e-4) per the code.
        """
        beamspace, H, SVD_rates = inputs

        with tf.GradientTape() as tape:
            # Full forward pass (all inside tape scope)
            SEhat, phat_ssb, _, bhat, _ = self.forward(beamspace, H)

            # Select ideal rate for each user's serving cell (Eq.31)
            SEtrue = select_svd_rate(SVD_rates, bhat)  # [B, U]

            # NBL loss: MSE between ideal and predicted achievable SE (Eq.32)
            loss = nbl_loss(SEtrue, SEhat)

        # Compute gradients w.r.t. both networks simultaneously
        all_variables = (
            self.ssb_network.trainable_variables +
            self.csi_network.trainable_variables
        )
        grads = tape.gradient(loss, all_variables)

        # Clip gradients to prevent exploding gradients (complex-valued system)
        grads, _ = tf.clip_by_global_norm(grads, clip_norm=5.0)

        # Apply with separate optimizers (will be set in compile())
        n_ssb = len(self.ssb_network.trainable_variables)
        self.ssb_optimizer.apply_gradients(
            zip(grads[:n_ssb], self.ssb_network.trainable_variables)
        )
        self.csi_optimizer.apply_gradients(
            zip(grads[n_ssb:], self.csi_network.trainable_variables)
        )

        # Update metrics
        self._loss_tracker.update_state(loss)
        self._ssb_rsrp_tracker.update_state(
            tf.reduce_mean(10.0 * tf.experimental.numpy.log10(phat_ssb + 1e-10))
        )

        return {
            'se_loss':    self._loss_tracker.result(),
            'ssb_rsrp_db': self._ssb_rsrp_tracker.result()
        }

    # -------------------------------------------------------------------------
    # WARMUP TRAINING STEP — SSB only, maximise RSRP (optional phase 1)
    # -------------------------------------------------------------------------

    def warmup_ssb_step(
        self,
        beamspace: tf.Tensor,
        H: tf.Tensor
    ) -> tf.Tensor:
        """
        Optional warmup: train only the SSB network to maximise RSRP.

        This does NOT use the SE loss — it just maximises signal power,
        giving the SSB network a good starting point before end-to-end training.
        The code's train_step1 does exactly this with loss = -mean(RSRP).

        Args:
            beamspace : [B, Nx1+1, Ny1+1, C*2*Lmax]
            H         : [B, C, U, Nr, Nt]

        Returns:
            ssb_loss : scalar RSRP loss (negative mean RSRP)
        """
        with tf.GradientTape() as tape:
            pred_ssb_beamspace = self.ssb_network(beamspace, training=True)
            ssb_codebook       = tf_deconvert_beamspace(pred_ssb_beamspace, Lmax=self.Lmax)
            phat, _, _         = tf_rsrp_rx_comb(H, ssb_codebook)
            ssb_loss           = -tf.reduce_mean(phat)   # maximise RSRP

        grads = tape.gradient(ssb_loss, self.ssb_network.trainable_variables)
        self.ssb_optimizer.apply_gradients(
            zip(grads, self.ssb_network.trainable_variables)
        )
        return ssb_loss

    # -------------------------------------------------------------------------
    # VALIDATION STEP
    # -------------------------------------------------------------------------

    def test_step(self, inputs: tuple) -> dict:
        """
        Validation step: same forward pass, no gradient update.
        """
        beamspace, H, SVD_rates = inputs

        # No GradientTape — inference only
        pred_ssb_beamspace = self.ssb_network(beamspace, training=False)
        ssb_codebook       = tf_deconvert_beamspace(pred_ssb_beamspace, Lmax=self.Lmax)
        phat_ssb, mhat, bhat = tf_rsrp_rx_comb(H, ssb_codebook)

        pred_csi_beamspace = self.csi_network(beamspace, training=False)
        csi_codebook       = tf_deconvert_beamspace_csi(pred_csi_beamspace, Lmax=self.Lmax, N=self.N_csi, bg=self.bg)
        csi_sub            = tf_select_subset(ssb_codebook, csi_codebook, mhat, bhat, self.Ncsi)
        _, ihat, _         = tf_rsrp_rx_comb_csistep(H, csi_sub, bhat)
        SEhat              = tf_se_mumimo(csi_sub, ihat, bhat, H)
        SEtrue             = select_svd_rate(SVD_rates, bhat)

        val_loss = nbl_loss(SEtrue, SEhat)
        self._loss_tracker.update_state(val_loss)
        return {'val_se_loss': self._loss_tracker.result()}

    # -------------------------------------------------------------------------
    # COMPILE — attach optimizers with separate learning rates
    # -------------------------------------------------------------------------

    def compile(
        self,
        ssb_lr: float = 4e-4,   # from code Cell 38
        csi_lr: float = 2e-4,   # from code Cell 38
        **kwargs
    ):
        """
        Attach two Adam optimizers — one per sub-network.

        Paper uses:
            SSB optimizer: Adam, lr = 4e-4
            CSI optimizer: Adam, lr = 2e-4
        """
        super().compile(**kwargs)
        self.ssb_optimizer = keras.optimizers.Adam(learning_rate=ssb_lr)
        self.csi_optimizer = keras.optimizers.Adam(learning_rate=csi_lr)

    @property
    def metrics(self):
        return [self._loss_tracker, self._ssb_rsrp_tracker]

    # -------------------------------------------------------------------------
    # INFERENCE — return codebooks for use by the network
    # -------------------------------------------------------------------------

    def predict_codebooks(
        self,
        beamspace: tf.Tensor
    ) -> tuple[tf.Tensor, tf.Tensor]:
        """
        Inference: given beamspace input, return SSB and CSI-RS codebooks.

        Args:
            beamspace : [B, Nx1+1, Ny1+1, C*2*Lmax]

        Returns:
            ssb_codebook : [B, C, Lmax, Nt]          complex64
            csi_codebook : [B, C, Lmax, N_csi, bg, Nt] complex64
        """
        pred_ssb = self.ssb_network(beamspace, training=False)
        pred_csi = self.csi_network(beamspace, training=False)
        ssb_codebook = tf_deconvert_beamspace(pred_ssb, Lmax=self.Lmax)
        csi_codebook = tf_deconvert_beamspace_csi(pred_csi, Lmax=self.Lmax, N=self.N_csi, bg=self.bg)
        return ssb_codebook, csi_codebook


# =============================================================================
# DATASET PIPELINE  (paper Section V)
# =============================================================================

def make_tf_dataset(
    beamspace_set: np.ndarray,
    channel_set:   np.ndarray,
    svd_rate_set:  np.ndarray,
    batch_size:    int  = BATCH_SIZE,
    shuffle:       bool = True,
    total_samples: int  = None
) -> tf.data.Dataset:
    """
    Build a tf.data.Dataset from pre-generated numpy arrays.

    Dataset structure per sample:
        beamspace : [Nx1+1, Ny1+1, C*2*Lmax]  float32
                    (Scaled to [0,1] by MinMaxScaler during peel_channels)
        channels  : [C, Umax, Nr, Nt]           complex64
                    (Zero-padded where U < Umax)
        svd_rates : [C, Umax]                   float32
                    (Zero where user is inactive)

    Args:
        beamspace_set  : shape [N, Nx1+1, Ny1+1, C*2*Lmax]
        channel_set    : shape [N, C, Umax, Nr, Nt]
        svd_rate_set   : shape [N, C, Umax]
        batch_size     : samples per batch
        shuffle        : whether to shuffle (True for train, False for val/test)
        total_samples  : truncate dataset to this many samples (None = use all)

    Returns:
        tf.data.Dataset yielding (beamspace, channels, svd_rates) batches
    """
    if total_samples is not None:
        beamspace_set = beamspace_set[:total_samples]
        channel_set   = channel_set[:total_samples]
        svd_rate_set  = svd_rate_set[:total_samples]

    AUTO = tf.data.AUTOTUNE

    ds_beamspace = tf.data.Dataset.from_tensor_slices(beamspace_set)
    ds_channels  = tf.data.Dataset.from_tensor_slices(channel_set)
    ds_svd_rates = tf.data.Dataset.from_tensor_slices(svd_rate_set)

    ds = tf.data.Dataset.zip((ds_beamspace, ds_channels, ds_svd_rates))

    if shuffle:
        ds = ds.shuffle(buffer_size=min(10_000, len(beamspace_set)))

    ds = ds.batch(batch_size, drop_remainder=True)   # drop_remainder keeps shapes static
    ds = ds.prefetch(AUTO)
    return ds


def load_dataset(filepath: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, list]:
    """
    Load a pre-generated dataset pickle from the comm team.

    Pickle format (from peel_channels / spare_peel_channels):
        [beamspace_set, channel_set, SVD_rate_set, scaler_set]

    Args:
        filepath : path to the .pickle file

    Returns:
        beamspace_set : [N, Nx1+1, Ny1+1, C*2*Lmax]   float32
        channel_set   : [N, C, Umax, Nr, Nt]            complex64
        svd_rate_set  : [N, C, Umax]                    float32
        scaler_set    : list of C MinMaxScaler objects (for inverse transform)
    """
    with open(filepath, 'rb') as f:
        beamspace_set, channel_set, svd_rate_set, scaler_set = pickle.load(f)
    return beamspace_set, channel_set, svd_rate_set, scaler_set


def split_dataset(
    beamspace_set: np.ndarray,
    channel_set:   np.ndarray,
    svd_rate_set:  np.ndarray,
    train_ratio:   float = 0.8,
    batch_size:    int   = BATCH_SIZE
) -> tuple[tf.data.Dataset, tf.data.Dataset]:
    """
    Split loaded data into train and validation tf.data.Datasets.

    Paper uses 80/20 split on 20,000 samples = 16,000 train / 4,000 val.

    Args:
        beamspace_set, channel_set, svd_rate_set : full dataset arrays
        train_ratio : fraction for training
        batch_size  : samples per batch

    Returns:
        train_ds : shuffled training dataset
        val_ds   : non-shuffled validation dataset
    """
    n_total = len(beamspace_set)
    n_train = int(n_total * train_ratio)

    train_ds = make_tf_dataset(
        beamspace_set[:n_train],
        channel_set[:n_train],
        svd_rate_set[:n_train],
        batch_size = batch_size,
        shuffle    = True
    )
    val_ds = make_tf_dataset(
        beamspace_set[n_train:],
        channel_set[n_train:],
        svd_rate_set[n_train:],
        batch_size = batch_size,
        shuffle    = False
    )
    return train_ds, val_ds


# =============================================================================
# TRAINING LOOP  (paper Section IV + code Cell 37)
# =============================================================================

def train_nbl(
    model:        NBL,
    train_ds:     tf.data.Dataset,
    val_ds:       tf.data.Dataset,
    n_epochs:     int = 100,
    checkpoint_dir: str = './checkpoints',
    save_every:   int = 5
) -> dict:
    """
    Full NBL training loop matching the paper's training procedure.

    Structure:
        For each epoch:
            For each batch in train_ds:
                1. Warmup SSB step (optional, first epoch only)
                2. Full end-to-end step (SSB + CSI jointly, Eq.32 loss)
            Validate on val_ds
            Save checkpoint every save_every epochs

    Args:
        model         : compiled NBL instance
        train_ds      : training tf.data.Dataset
        val_ds        : validation tf.data.Dataset
        n_epochs      : number of training epochs
        checkpoint_dir: directory to save model checkpoints
        save_every    : checkpoint frequency in epochs

    Returns:
        history : dict with 'train_loss', 'val_loss', 'ssb_rsrp' per epoch
    """
    os.makedirs(checkpoint_dir, exist_ok=True)

    history = {
        'train_se_loss': [],
        'val_se_loss':   [],
        'ssb_rsrp_db':   []
    }

    for epoch in range(n_epochs):
        print(f"\nEpoch {epoch+1}/{n_epochs}")

        # --- Training ---
        train_losses   = []
        train_rsrp_dbs = []

        for batch_idx, (beamspace, H, SVD_rates) in enumerate(train_ds):



            # End-to-end training step (main contribution, Eq.32)
            metrics = model.train_step((beamspace, H, SVD_rates))
            train_losses.append(metrics['se_loss'].numpy())
            train_rsrp_dbs.append(metrics['ssb_rsrp_db'].numpy())

            if batch_idx % 20 == 0:
                print(
                    f"  batch {batch_idx}: "
                    f"SE_loss={train_losses[-1]:.4f}  "
                    f"SSB_RSRP={train_rsrp_dbs[-1]:.1f}dBm"
                )

        # --- Validation ---
        val_losses = []
        for beamspace, H, SVD_rates in val_ds:
            val_metrics = model.test_step((beamspace, H, SVD_rates))
            val_losses.append(val_metrics['val_se_loss'].numpy())

        mean_train = np.mean(train_losses)
        mean_val   = np.mean(val_losses)
        mean_rsrp  = np.mean(train_rsrp_dbs)

        history['train_se_loss'].append(mean_train)
        history['val_se_loss'].append(mean_val)
        history['ssb_rsrp_db'].append(mean_rsrp)

        print(
            f"  -> train_loss={mean_train:.4f}  "
            f"val_loss={mean_val:.4f}  "
            f"ssb_rsrp={mean_rsrp:.1f}dBm"
        )

        # Reset metric states between epochs
        for m in model.metrics:
            m.reset_state()

        # --- Checkpoint ---
        if (epoch + 1) % save_every == 0:
            ckpt_ssb = os.path.join(checkpoint_dir, f'ssb_epoch_{epoch+1}.weights.h5')
            ckpt_csi = os.path.join(checkpoint_dir, f'csi_epoch_{epoch+1}.weights.h5')
            model.ssb_network.save_weights(ckpt_ssb)
            model.csi_network.save_weights(ckpt_csi)
            print(f"  Saved checkpoints: {ckpt_ssb}, {ckpt_csi}")

    return history


def save_model(model: NBL, save_dir: str, scene_id: str, n_samples: int):
    """
    Save trained SSB and CSI networks to disk (matching code Cell 44 format).

    Args:
        model     : trained NBL instance
        save_dir  : base directory for saved models
        scene_id  : scene identifier ('A' or 'B')
        n_samples : number of training samples used
    """
    os.makedirs(save_dir, exist_ok=True)
    ssb_path = os.path.join(
        save_dir,
        f'scene_{scene_id}_{n_samples}_ssb_{Nx}_{Ny}_{Nx1}_{Ny1}_{Lmax}.keras'
    )
    csi_path = os.path.join(
        save_dir,
        f'scene_{scene_id}_{n_samples}_csi_{Nx}_{Ny}_{Nx1}_{Ny1}_{Lmax}.keras'
    )
    model.ssb_network.save(ssb_path)
    model.csi_network.save(csi_path)
    print(f"Saved SSB network -> {ssb_path}")
    print(f"Saved CSI network -> {csi_path}")


def load_model(save_dir: str, scene_id: str, n_samples: int) -> NBL:
    """
    Load a trained NBL model from saved Keras files.

    Args:
        save_dir  : base directory where models are saved
        scene_id  : scene identifier ('A' or 'B')
        n_samples : number of training samples used

    Returns:
        model : NBL instance with loaded weights
    """
    ssb_path = os.path.join(
        save_dir,
        f'scene_{scene_id}_{n_samples}_ssb_{Nx}_{Ny}_{Nx1}_{Ny1}_{Lmax}.keras'
    )
    csi_path = os.path.join(
        save_dir,
        f'scene_{scene_id}_{n_samples}_csi_{Nx}_{Ny}_{Nx1}_{Ny1}_{Lmax}.keras'
    )
    ssb_net = keras.models.load_model(ssb_path)
    csi_net = keras.models.load_model(csi_path)

    model = NBL(ssb_network=ssb_net, csi_network=csi_net)
    model.compile()
    print(f"Loaded NBL model from {save_dir}")
    return model


# =============================================================================
# EVALUATION  (paper Section VI)
# =============================================================================

def evaluate_ssb(
    model:    NBL,
    test_ds:  tf.data.Dataset
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Evaluate SSB codebook performance (paper Figure 5, 6).

    Computes RSRP for all users across test batches.

    Args:
        model   : trained NBL instance
        test_ds : test dataset

    Returns:
        nbl_rsrp : RSRP with NBL codebooks,  shape [N_test * Umax]  float32
        nbl_m    : SSB beam indices,          shape [N_test * Umax]  int32
        nbl_b    : cell assignments,          shape [N_test * Umax]  int32
    """
    all_rsrp, all_m, all_b = [], [], []

    for beamspace, H, _ in test_ds:
        # Narrow the channel to single time-frequency resource for SSB
        H_ssb = H[:, :, :, 0, 0, :, :] if H.ndim == 7 else H

        pred_ssb = model.ssb_network(beamspace, training=False)
        ssb_codebook = tf_deconvert_beamspace(pred_ssb, Lmax=model.Lmax)
        p, m, b = tf_rsrp_rx_comb(H_ssb, ssb_codebook)

        all_rsrp.append(p.numpy())
        all_m.append(m.numpy())
        all_b.append(b.numpy())

    nbl_rsrp = np.concatenate(all_rsrp).flatten()
    nbl_m    = np.concatenate(all_m).flatten()
    nbl_b    = np.concatenate(all_b).flatten()

    # Filter out inactive users (zero RSRP = padded slot)
    active = nbl_rsrp > 0
    return nbl_rsrp[active], nbl_m[active], nbl_b[active]


def evaluate_csi(
    model:   NBL,
    test_ds: tf.data.Dataset,
    Ncsi:    int = NCSI
) -> tuple[np.ndarray, np.ndarray]:
    """
    Evaluate CSI-RS performance (paper Figure 7, 8, 9).

    Returns achievable SE and ideal SVD rate for all active users
    across test batches.

    Args:
        model   : trained NBL instance
        test_ds : test dataset
        Ncsi    : CSI-RS subset size

    Returns:
        SEhat   : predicted achievable SE, [N_active] float32
        SEtrue  : ideal SVD rate,          [N_active] float32
    """
    all_SEhat, all_SEtrue = [], []

    for beamspace, H, SVD_rates in test_ds:
        H_narrow = H[:, :, :, 0, 0, :, :] if H.ndim == 7 else H

        pred_ssb      = model.ssb_network(beamspace, training=False)
        ssb_codebook  = tf_deconvert_beamspace(pred_ssb, Lmax=model.Lmax)
        _, mhat, bhat = tf_rsrp_rx_comb(H_narrow, ssb_codebook)

        pred_csi     = model.csi_network(beamspace, training=False)
        csi_codebook = tf_deconvert_beamspace_csi(
            pred_csi, Lmax=model.Lmax, N=model.N_csi, bg=model.bg)
        csi_sub      = tf_select_subset(
            ssb_codebook, csi_codebook, mhat, bhat, Ncsi)
        _, ihat, _   = tf_rsrp_rx_comb_csistep(H_narrow, csi_sub, bhat)

        SEhat  = tf_se_mumimo(csi_sub, ihat, bhat, H_narrow)
        SEtrue = select_svd_rate(SVD_rates, bhat)

        all_SEhat.append(SEhat.numpy())
        all_SEtrue.append(SEtrue.numpy())

    SEhat_flat  = np.concatenate(all_SEhat).flatten()
    SEtrue_flat = np.concatenate(all_SEtrue).flatten()

    active = SEtrue_flat > 0.1
    return SEhat_flat[active], SEtrue_flat[active]


def evaluate_esse(
    model:   NBL,
    test_ds: tf.data.Dataset,
    Ncsi:    int = NCSI
) -> tuple[np.ndarray, np.ndarray]:
    """
    Evaluate full network Effective Sum Spectral Efficiency (paper Eq.25, Figure 10-13).

    This runs the complete beam management -> RZF precoding -> data transmission
    pipeline and returns per-sample ESSE for NBL and DFT comparison.

    Note: tf_get_RZF uses numpy (numba-accelerated) so this step breaks out of
    the TF graph — that is intentional and matches the paper's design (RZF is
    evaluation-only, not in the training gradient path).

    Args:
        model   : trained NBL instance
        test_ds : test dataset (uses full spatio-temporal channel H)
        Ncsi    : CSI-RS subset size

    Returns:
        nbl_esse : per-sample network ESSE with NBL codebooks, [N_test]  float32
        dft_esse : per-sample network ESSE with DFT codebooks, [N_test]  float32
    """
    # Build DFT codebooks for comparison
    dft_ssb_cb = DFT_codebook(n_beams=Lmax,           Nx=Nx, Ny=Ny, OH=2,     OV=2)

    csi_codebook_dft = gen_codebook(Nx, Ny, OH=2, OV=1)
    dft_csi_cb = np.tile(csi_codebook_dft.reshape(1, 1, Lmax, 8, 4, Nt), [batch_size, C, 1, 1, 1, 1])

# SSB: shape (Lmax, Nt, 1) -> (1, 1, Lmax, Nt) -> tile to [B, C, Lmax, Nt]
    dft_ssb_tiled = np.tile(
    dft_ssb_cb.reshape(1, 1, Lmax, Nt),
    [BATCH_SIZE, C, 1, 1]
    ) / np.sqrt(Nt)

# CSI: shape (Lmax*N_csi, Nt, 1) -> (Lmax, N_csi, Nt)
#      then add Bg axis by tiling -> (Lmax, N_csi, Bg, Nt)
#      then tile to [B, C, Lmax, N_csi, Bg, Nt]
    dft_csi_flat = dft_csi_cb.reshape(Lmax, N_csi, Nt)               # 32 768 elems
    dft_csi_bg   = np.tile(
    dft_csi_flat[:, :, np.newaxis, :],                            # (Lmax, N_csi, 1, Nt)
    [1, 1, Bg, 1]                                                 # (Lmax, N_csi, Bg, Nt)
    )                                                                 # 131 072 elems
    dft_csi_tiled = np.tile(
    dft_csi_bg.reshape(1, 1, Lmax, N_csi, Bg, Nt),
    [BATCH_SIZE, C, 1, 1, 1, 1]
    ).astype(ctype)

    nbl_esse_list = []
    dft_esse_list = []

    for beamspace, H_full, SVD_rates in test_ds:
        # Use time-freq averaged channel for beam selection (SSB/CSI-RS)
        H_narrow = H_full[:, :, :, 0, 0, :, :]   # [B, C, U, Nr, Nt]

        # --- NBL path ---
        pred_ssb     = model.ssb_network(beamspace, training=False)
        ssb_codebook = tf_deconvert_beamspace(pred_ssb, Lmax=model.Lmax)
        _, mhat, bhat = tf_rsrp_rx_comb(H_narrow, ssb_codebook)

        pred_csi     = model.csi_network(beamspace, training=False)
        csi_codebook = tf_deconvert_beamspace_csi(pred_csi, Lmax=model.Lmax, N=model.N_csi, bg=model.bg)
        csi_sub      = tf_select_subset(ssb_codebook, csi_codebook, mhat, bhat, Ncsi)
        _, ihat, _   = tf_rsrp_rx_comb_csistep(H_narrow, csi_sub, bhat)

        # Apply analog precoder to full (T, K) channel
        Heff_nbl   = tf_apply_chan_F(csi_sub, ihat, bhat, H_full)   # [B, U, C, T, K, Nr, Bg]
        F_rzf_nbl  = tf_get_RZF(Heff_nbl.numpy())                   # RZF in numpy/numba
        esse_nbl   = tf.reduce_mean(
            tf_MU_MIMO_Opt(Heff_nbl, F_rzf_nbl, bhat),
            axis=[1, 2]                                              # mean over T, K
        )
        nbl_esse_list.append(esse_nbl.numpy())

        # --- DFT baseline path ---
        _, m_dft, b_dft = tf_rsrp_rx_comb(H_narrow, tf.constant(dft_ssb_tiled, dtype=tf.complex64))
        csi_sub_dft     = tf_select_subset(
            tf.constant(dft_ssb_tiled, dtype=tf.complex64),
            tf.constant(dft_csi_tiled, dtype=tf.complex64),
            m_dft, b_dft, Ncsi
        )
        _, ihat_dft, _  = tf_rsrp_rx_comb_csistep(H_narrow, csi_sub_dft, b_dft)
        Heff_dft        = tf_apply_chan_F(csi_sub_dft, ihat_dft, b_dft, H_full)
        F_rzf_dft       = tf_get_RZF(Heff_dft.numpy())
        esse_dft        = tf.reduce_mean(
            tf_MU_MIMO_Opt(Heff_dft, F_rzf_dft, b_dft),
            axis=[1, 2]
        )
        dft_esse_list.append(esse_dft.numpy())

    nbl_esse = np.concatenate(nbl_esse_list).flatten()
    dft_esse = np.concatenate(dft_esse_list).flatten()
    return nbl_esse, dft_esse


# =============================================================================
# ENTRY POINT — wire everything together
# =============================================================================

def main():
    """
    Main entry point: load data -> build model -> train -> evaluate.
    Matches the paper's experimental setup from Section V-VI.
    """

    # --- Configuration ---
    DATASET_PATH  = '/kaggle/input/datasets/khaledalaskari/munich-dataset/6pg_scene_A_20000_inputs_outputs_16_16_16_16_16.pickle'
    MODEL_DIR     = '/content/nbl_eval'
    SCENE_ID      = 'A'
    N_SAMPLES     = 20000
    N_EPOCHS      = 100

    # --- Load dataset (generated by comm team using Sionna) ---
    print("Loading dataset...")
    beamspace_set, channel_set, svd_rate_set, scaler_set = load_dataset(DATASET_PATH)
    print(f"  beamspace: {beamspace_set.shape}")   # [20000, Nx1+1, Ny1+1, C*2*Lmax]
    print(f"  channels:  {channel_set.shape}")      # [20000, C, Umax, Nr, Nt]
    print(f"  svd_rates: {svd_rate_set.shape}")     # [20000, C, Umax]

    # --- Split into train / validation ---
    train_ds, val_ds = split_dataset(
        beamspace_set, channel_set, svd_rate_set,
        train_ratio = 0.8,
        batch_size  = BATCH_SIZE
    )
    print(f"Train batches: {train_ds.cardinality().numpy()}")
    print(f"Val batches:   {val_ds.cardinality().numpy()}")

    # --- Build and compile model ---
    print("\nBuilding NBL model...")
    model = NBL(
        Lmax       = Lmax,
        N_csi      = N_csi,
        bg         = Bg,
        Ncsi       = NCSI,
        batch_size = BATCH_SIZE
    )
    model.compile(ssb_lr=4e-4, csi_lr=2e-4)
    model.ssb_network.summary()
    model.csi_network.summary()

    # --- Train ---
    print("\nStarting training...")
    history = train_nbl(
        model,
        train_ds,
        val_ds,
        n_epochs       = N_EPOCHS,
        checkpoint_dir = os.path.join(MODEL_DIR, 'checkpoints'),
        save_every     = 2
    )

    # --- Save final model ---
    save_model(model, MODEL_DIR, SCENE_ID, N_SAMPLES)

    # --- Evaluate on test set ---
    # Load test set (separate from training data — unseen by the model)
    TEST_PATH = '/teamspace/studios/this_studio/outputs/eval_6pg_scene_A_2000_inputs_outputs_16_16_16_16_16.pickle'
    bs_test, ch_test, svd_test, _ = load_dataset(TEST_PATH)
    _, test_ds = split_dataset(bs_test, ch_test, svd_test, train_ratio=0.0, batch_size=BATCH_SIZE)

    print("\nEvaluating SSB performance...")
    nbl_rsrp, _, _ = evaluate_ssb(model, test_ds)
    print(f"  Mean NBL RSRP: {10*np.log10(nbl_rsrp.mean() + 1e-10):.1f} dBm")

    print("\nEvaluating CSI-RS performance...")
    SEhat, SEtrue, S, INR = evaluate_csi(model, test_ds)
    print(f"  Mean SE gap from ideal: {(SEtrue - SEhat).mean():.2f} bps/Hz")

    print("\nEvaluating ESSE (full network)...")
    nbl_esse, dft_esse = evaluate_esse(model, test_ds)
    gain = (nbl_esse.sum(axis=-1) - dft_esse.sum(axis=-1)).mean()
    print(f"  Mean ESSE gain over DFT: {gain:.1f} bps/Hz")

    return model, history

"""
if __name__ == '__main__':
    main()
    """

"\nif __name__ == '__main__':\n    main()\n    "

## 16 · Keras 3 Compatibility Patch

In [23]:
# ═══════════════════════════════════════════════════════════════════════════════
# Keras 3 / TF 2.16+ compatibility patch for NBL training methods
# ═══════════════════════════════════════════════════════════════════════════════

def _train_step_k3(self, inputs):
    beamspace, H, SVD_rates = inputs

    # ── Stage 1: SSB on RSRP loss (its own tape) ───────────────────────────
    ssb_vars = self.ssb_network.trainable_variables
    with tf.GradientTape() as ssb_tape:
        pred_ssb       = self.ssb_network(beamspace, training=True)
        ssb_codebook   = tf_deconvert_beamspace(pred_ssb, Lmax=self.Lmax)
        phat_ssb, _, _ = tf_rsrp_rx_comb(H, ssb_codebook)
        ssb_loss       = -tf.reduce_mean(phat_ssb)
    g = ssb_tape.gradient(ssb_loss, ssb_vars)
    g = [gi if gi is not None else tf.zeros_like(v)
         for gi, v in zip(g, ssb_vars)]
    g, _ = tf.clip_by_global_norm(g, 5.0)
    self.ssb_optimizer.apply(g, ssb_vars)

    # ── Stage 2: CSI on SE loss ─────────────────────────────────────────────
    # Reuse ssb_codebook from Stage 1 — no second SSB forward pass needed.
    # stop_gradient makes it a constant from the CSI tape's perspective.
    ssb_codebook_sg        = tf.stop_gradient(ssb_codebook)
    phat_metric, mhat_sg, bhat_sg = tf_rsrp_rx_comb(H, ssb_codebook_sg)
    mhat_sg  = tf.stop_gradient(mhat_sg)
    bhat_sg  = tf.stop_gradient(bhat_sg)

    # The compiled function contains the tape + tf_select_subset together,
    # so TF traces through the selection into csi_codebook → gradients flow.
    csi_loss, grads = _csi_train_stage(
        self.csi_network, beamspace, H, SVD_rates,
        ssb_codebook_sg, mhat_sg, bhat_sg,
        self.Ncsi, self.Lmax, self.N_csi, self.bg
    )
    csi_vars = self.csi_network.trainable_variables
    grads = [gi if gi is not None else tf.zeros_like(v)
             for gi, v in zip(grads, csi_vars)]
    grads, _ = tf.clip_by_global_norm(grads, 5.0)
    self.csi_optimizer.apply(grads, csi_vars)

    # ── Metrics ────────────────────────────────────────────────────────────
    self._loss_tracker.update_state(csi_loss)
    self._ssb_rsrp_tracker.update_state(
        tf.reduce_mean(10.0 * tf.experimental.numpy.log10(
            tf.math.real(phat_metric) + 1e-10)))
    return {'se_loss':     self._loss_tracker.result(),
            'ssb_rsrp_db': self._ssb_rsrp_tracker.result()}

NBL.train_step = _train_step_k3


def _warmup_ssb_step_k3(self, beamspace, H):
    ssb_vars = self.ssb_network.trainable_variables
    with tf.GradientTape() as tape:
        #tape.watch(ssb_vars)
        pred       = self.ssb_network(beamspace, training=True)
        cb         = tf_deconvert_beamspace(pred, Lmax=self.Lmax)
        phat, _, _ = tf_rsrp_rx_comb(H, cb)
        loss       = -tf.reduce_mean(phat)

    grads = tape.gradient(loss, ssb_vars)
    grads = [g if g is not None else tf.zeros_like(v)
             for g, v in zip(grads, ssb_vars)]
    self.ssb_optimizer.apply(grads, ssb_vars)
    return loss


def _test_step_k3(self, inputs):
    beamspace, H, SVD_rates = inputs
    pred_ssb      = self.ssb_network(beamspace, training=False)
    ssb_cb        = tf_deconvert_beamspace(pred_ssb, Lmax=self.Lmax)
    _, mhat, bhat = tf_rsrp_rx_comb(H, ssb_cb)
    pred_csi      = self.csi_network(beamspace, training=False)
    csi_cb        = tf_deconvert_beamspace_csi(pred_csi, Lmax=self.Lmax,
                                                N=self.N_csi, bg=self.bg)
    csi_sub       = tf_select_subset(ssb_cb, csi_cb, mhat, bhat, self.Ncsi)
    _, ihat, _    = tf_rsrp_rx_comb_csistep(H, csi_sub, bhat)
    SEhat         = tf_se_mumimo(csi_sub, ihat, bhat, H)
    SEtrue        = select_svd_rate(SVD_rates, bhat)
    val_loss      = nbl_loss(SEtrue, SEhat)
    self._loss_tracker.update_state(val_loss)
    return {'val_se_loss': self._loss_tracker.result()}


NBL.train_step      = _train_step_k3
NBL.warmup_ssb_step = _warmup_ssb_step_k3
NBL.test_step       = _test_step_k3

print("NBL training methods patched for Keras 3 / TF 2.16+  ✓")

NBL training methods patched for Keras 3 / TF 2.16+  ✓


---
# Part 2 · NBL Model Training

The cells below define the **Network Beamspace Learning (NBL)** model and train it on the
dataset generated in Part 1.  All dataset paths are taken directly from the variables
`train_path` and `eval_path` set in §13 above — no manual path editing needed.

> **Note on function shadowing:** `DFT_codebook`, `gen_codebook`, and `load_dataset` are
> redefined below with model-specific signatures.  The data-generation versions (Part 1)
> have already been executed and their outputs are saved to disk, so there is no conflict.


## 18 · Load Generated Dataset

`DATASET_PATH` and `TEST_PATH` are derived directly from the `train_path` / `eval_path`
variables produced in §13 above, so they always match the files on disk.


In [ ]:
# ── Bridge: link model section to data-generation outputs ────────────────────
NCSI         = 16              # total CSI-RS beams per cell (Lmax subset, paper §IV)
DATASET_PATH = train_path      # e.g. ./outputs/6pg_scene_A_20000_inputs_outputs_...pickle
MODEL_DIR    = os.path.join(OUTPUT_DIR, 'models')
SCENE_ID     = SCENE
N_SAMPLES    = DATASET_SIZE_TRAIN
N_EPOCHS     = 50

os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Training dataset : {DATASET_PATH}')
print(f'Model directory  : {MODEL_DIR}')

# Load training dataset (generated by Part 1 above)
print('\nLoading training dataset...')
beamspace_set, channel_set, svd_rate_set, scaler_set = load_dataset(DATASET_PATH)
print(f'  beamspace : {beamspace_set.shape}')  # [N, Nx1+1, Ny1+1, C*2*Lmax]
print(f'  channels  : {channel_set.shape}')    # [N, C, Umax, Nr, Nt]
print(f'  svd_rates : {svd_rate_set.shape}')   # [N, C, Umax]


## 19 · Build Train / Validation Datasets

80 / 20 split → `train_ds` and `val_ds` as `tf.data.Dataset` objects.


In [ ]:
train_ds, val_ds = split_dataset(
    beamspace_set, channel_set, svd_rate_set,
    train_ratio = 0.8,
    batch_size  = BATCH_SIZE
)
print(f'Train batches: {train_ds.cardinality().numpy()}')
print(f'Val   batches: {val_ds.cardinality().numpy()}')


## 20 · Build, Compile & Train the NBL Model

Two-phase training (paper §IV):
1. **Warmup** (epoch 0 only): SSB network maximises RSRP.
2. **End-to-end**: Both networks trained jointly with the SE-MSE loss (Eq. 32).

Checkpoints saved every 2 epochs to `MODEL_DIR/checkpoints/`.


In [ ]:
# Build and compile the NBL model
print('Building NBL model...')
model = NBL(
    Lmax       = Lmax,
    N_csi      = N_csi,
    bg         = Bg,
    Ncsi       = NCSI,
    batch_size = BATCH_SIZE
)
model.compile(ssb_lr=4e-4, csi_lr=2e-4)
model.ssb_network.summary()
model.csi_network.summary()

# Train
print('\nStarting training...')
history = train_nbl(
    model,
    train_ds,
    val_ds,
    n_epochs       = N_EPOCHS,
    checkpoint_dir = os.path.join(MODEL_DIR, 'checkpoints'),
    save_every     = 2
)


## 21 · Save Trained Model

Saves SSB and CSI Keras models to `MODEL_DIR` using the same naming convention as the paper.


In [ ]:
save_model(model, MODEL_DIR, SCENE_ID, N_SAMPLES)

In [ ]:
model = NBL(Lmax=Lmax, N_csi=N_csi, bg=Bg, Ncsi=NCSI, batch_size=BATCH_SIZE)
model.compile(ssb_lr=4e-4, csi_lr=2e-4)
model.ssb_network.load_weights('outputs/ssb_epoch_34.weights.h5')
model.csi_network.load_weights('outputs/csi_epoch_34.weights.h5')
# Then save as .keras if you want
save_model(model, MODEL_DIR, SCENE_ID, N_SAMPLES)